In [1]:
# ============================================================
# CELL 1 — BIGBIRD + EMOTION-BASED
# ============================================================

import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("BIGBIRD + EMOTION-BASED")
print("=" * 60)

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

MODEL_NAME = "google/bigbird-roberta-base"
EXPERIMENT_NAME = "BigBird + Emotion-Based"
CLEANING_METHOD = "Emotion-Based"

MAX_LENGTH = 512
METADATA_DIM = 53
FREEZE_TRANSFORMER = True

BASE_RESULT_DIR = (
    "/kaggle/working/"
    "bigbird_emotion"
)

RESULT_DIR = os.path.join(
    BASE_RESULT_DIR,
    "results"
)

CHECKPOINT_DIR = os.path.join(
    BASE_RESULT_DIR,
    "checkpoints"
)

BACKUP_DIR = os.path.join(
    BASE_RESULT_DIR,
    "backup"
)

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

EMOTION_MODEL_NAME = (
    "cardiffnlp/"
    "twitter-roberta-base-emotion-latest"
)

EMOTION_LABELS = [
    "anger",
    "anticipation",
    "disgust",
    "fear",
    "joy",
    "love",
    "optimism",
    "pessimism",
    "sadness",
    "surprise",
    "trust"
]

assert len(EMOTION_LABELS) == 11

print()
print("Model:", MODEL_NAME)
print("Cleaning:", CLEANING_METHOD)
print("Context length:", MAX_LENGTH)
print("Metadata features:", METADATA_DIM)
print("Transformer frozen:", FREEZE_TRANSFORMER)
print("Seed:", SEED)

print()
print("=" * 60)
print("EMOTION MODEL")
print("=" * 60)

print(
    "Emotion model:",
    EMOTION_MODEL_NAME
)

print()
print("Emotion labels:")

for i, label in enumerate(
    EMOTION_LABELS,
    start=1
):
    print(
        f"{i:02d}. {label}"
    )

print()
print("Number of emotion features:", 11)

print()
print("=" * 60)
print("BIGBIRD CONFIGURATION")
print("=" * 60)

print("Base encoder: BigBird-RoBERTa")
print("Pretrained model:", MODEL_NAME)
print("Backbone training: FROZEN")
print("Downstream classifier: TRAINABLE")
print("Metadata branch: TRAINABLE")
print("Emotion features: 11")

config = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "cleaning_method": CLEANING_METHOD,
    "context_length": MAX_LENGTH,
    "metadata_features": METADATA_DIM,
    "transformer_frozen": FREEZE_TRANSFORMER,
    "seed": SEED,
    "emotion_model": EMOTION_MODEL_NAME,
    "emotion_labels": EMOTION_LABELS
}

with open(
    os.path.join(
        RESULT_DIR,
        "experiment_config.json"
    ),
    "w"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print()
print("=" * 60)
print("CELL 1 COMPLETE")
print("=" * 60)

print("✓ BigBird selected")
print("✓ Emotion-Based cleaning selected")
print("✓ Emotion model locked")
print("✓ 11 emotion labels defined")
print("✓ Context length = 512")
print("✓ Metadata dimension = 53")
print("✓ Transformer backbone = FROZEN")
print("✓ Downstream layers = TRAINABLE")
print("✓ Random seed = 42")
print("✓ Result directory created")
print("✓ Checkpoint directory created")
print("✓ Backup directory created")
print("✓ Experiment configuration saved")
print("✓ Previous experiment outputs untouched")

BIGBIRD + EMOTION-BASED

Device: cuda
GPU: Tesla T4
CUDA: 12.8

Model: google/bigbird-roberta-base
Cleaning: Emotion-Based
Context length: 512
Metadata features: 53
Transformer frozen: True
Seed: 42

EMOTION MODEL
Emotion model: cardiffnlp/twitter-roberta-base-emotion-latest

Emotion labels:
01. anger
02. anticipation
03. disgust
04. fear
05. joy
06. love
07. optimism
08. pessimism
09. sadness
10. surprise
11. trust

Number of emotion features: 11

BIGBIRD CONFIGURATION
Base encoder: BigBird-RoBERTa
Pretrained model: google/bigbird-roberta-base
Backbone training: FROZEN
Downstream classifier: TRAINABLE
Metadata branch: TRAINABLE
Emotion features: 11

CELL 1 COMPLETE
✓ BigBird selected
✓ Emotion-Based cleaning selected
✓ Emotion model locked
✓ 11 emotion labels defined
✓ Context length = 512
✓ Metadata dimension = 53
✓ Transformer backbone = FROZEN
✓ Downstream layers = TRAINABLE
✓ Random seed = 42
✓ Result directory created
✓ Checkpoint directory created
✓ Backup directory created
✓ Ex

In [2]:
# ============================================================
# CELL 2 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — LOADING EXACT USER-LEVEL SPLIT")
print("=" * 60)

SPLIT_ROOT = (
    "/kaggle/input/datasets/"
    "vedantnawghare35/"
    "mddl-experiment-1-splits"
)

TRAIN_PATH = os.path.join(
    SPLIT_ROOT,
    "MDDL_train_raw.pkl"
)

VAL_PATH = os.path.join(
    SPLIT_ROOT,
    "MDDL_val_raw.pkl"
)

TEST_PATH = os.path.join(
    SPLIT_ROOT,
    "MDDL_test_raw.pkl"
)

print()
print("TRAIN:", TRAIN_PATH)
print("VAL:  ", VAL_PATH)
print("TEST: ", TEST_PATH)

assert os.path.exists(TRAIN_PATH)
assert os.path.exists(VAL_PATH)
assert os.path.exists(TEST_PATH)

print()
print("✓ All split files found")

train_df = pd.read_pickle(TRAIN_PATH)
val_df = pd.read_pickle(VAL_PATH)
test_df = pd.read_pickle(TEST_PATH)

print()
print("=" * 60)
print("SPLIT SHAPES")
print("=" * 60)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

assert train_df.shape == (6135, 82)
assert val_df.shape == (767, 82)
assert test_df.shape == (767, 82)

print()
print("=" * 60)
print("REQUIRED COLUMN CHECK")
print("=" * 60)

required_columns = [
    "user_id",
    "username",
    "label",
    "timeline_text"
]

for column in required_columns:
    assert column in train_df.columns
    assert column in val_df.columns
    assert column in test_df.columns
    print("✓", column)

print()
print("=" * 60)
print("USER-LEVEL INTEGRITY")
print("=" * 60)

train_users = set(train_df["user_id"])
val_users = set(val_df["user_id"])
test_users = set(test_df["user_id"])

print(
    "Train unique users:",
    train_df["user_id"].nunique()
)

print(
    "Validation unique users:",
    val_df["user_id"].nunique()
)

print(
    "Test unique users:",
    test_df["user_id"].nunique()
)

assert train_df["user_id"].is_unique
assert val_df["user_id"].is_unique
assert test_df["user_id"].is_unique

print(
    "TRAIN ↔ VAL:",
    len(train_users & val_users)
)

print(
    "TRAIN ↔ TEST:",
    len(train_users & test_users)
)

print(
    "VAL ↔ TEST:",
    len(val_users & test_users)
)

assert len(train_users & val_users) == 0
assert len(train_users & test_users) == 0
assert len(val_users & test_users) == 0

print()
print("✓ No duplicate users")
print("✓ No cross-split overlap")

print()
print("=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

print()
print("TRAIN:")
print(
    train_df["label"]
    .value_counts()
    .sort_index()
)

print()
print("VALIDATION:")
print(
    val_df["label"]
    .value_counts()
    .sort_index()
)

print()
print("TEST:")
print(
    test_df["label"]
    .value_counts()
    .sort_index()
)

assert (
    train_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    {0: 4122, 1: 2013}
)

assert (
    val_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    {0: 515, 1: 252}
)

assert (
    test_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    {0: 516, 1: 251}
)

print()
print("=" * 60)
print("TIMELINE TEXT CHECK")
print("=" * 60)

train_null = int(
    train_df["timeline_text"].isna().sum()
)

val_null = int(
    val_df["timeline_text"].isna().sum()
)

test_null = int(
    test_df["timeline_text"].isna().sum()
)

print(
    "TRAIN null timeline texts:",
    train_null
)

print(
    "VALIDATION null timeline texts:",
    val_null
)

print(
    "TEST null timeline texts:",
    test_null
)

assert train_null == 0
assert val_null == 0
assert test_null == 0

print()
print("=" * 60)
print("CELL 2 COMPLETE")
print("=" * 60)

print("✓ Exact user-level split restored")
print("✓ TRAIN = 6135")
print("✓ VALIDATION = 767")
print("✓ TEST = 767")
print("✓ Same labels as previous experiments")
print("✓ No duplicate users")
print("✓ No cross-split user overlap")
print("✓ Timeline text available")

BIGBIRD + EMOTION-BASED — LOADING EXACT USER-LEVEL SPLIT

TRAIN: /kaggle/input/datasets/vedantnawghare35/mddl-experiment-1-splits/MDDL_train_raw.pkl
VAL:   /kaggle/input/datasets/vedantnawghare35/mddl-experiment-1-splits/MDDL_val_raw.pkl
TEST:  /kaggle/input/datasets/vedantnawghare35/mddl-experiment-1-splits/MDDL_test_raw.pkl

✓ All split files found

SPLIT SHAPES
Train: (6135, 82)
Validation: (767, 82)
Test: (767, 82)

REQUIRED COLUMN CHECK
✓ user_id
✓ username
✓ label
✓ timeline_text

USER-LEVEL INTEGRITY
Train unique users: 6135
Validation unique users: 767
Test unique users: 767
TRAIN ↔ VAL: 0
TRAIN ↔ TEST: 0
VAL ↔ TEST: 0

✓ No duplicate users
✓ No cross-split overlap

LABEL DISTRIBUTION

TRAIN:
label
0    4122
1    2013
Name: count, dtype: int64

VALIDATION:
label
0    515
1    252
Name: count, dtype: int64

TEST:
label
0    516
1    251
Name: count, dtype: int64

TIMELINE TEXT CHECK
TRAIN null timeline texts: 0
VALIDATION null timeline texts: 0
TEST null timeline texts: 0

CELL 

In [3]:
# ============================================================
# CELL 3 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR DETECTION")
print("=" * 60)

import os
import re
import json
import glob
from collections import defaultdict

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "vedantnawghare35/"
    "depression-detection-mddl-final-pbl/"
    "Dataset_MDDL"
)

D1_TIMELINE_DIR = os.path.join(
    DATASET_ROOT,
    "labeled",
    "positive",
    "data",
    "timeline"
)

assert os.path.exists(D1_TIMELINE_DIR)

d1_timeline_files = glob.glob(
    os.path.join(
        D1_TIMELINE_DIR,
        "*.json"
    )
)

print()
print(
    "D1 timeline files found:",
    len(d1_timeline_files)
)

assert len(d1_timeline_files) == 2626

REFINED_ANCHOR_PATTERNS = [
    r"\bi(?:'m| am)\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi\s+(?:was|were)\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi(?:'ve| have)\s+been\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi(?:'m| am)\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi\s+(?:was|were)\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi(?:'ve| have)\s+been\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi\s+have\s+(?:major\s+)?depression\b",
    r"\bi\s+suffer\s+from\s+(?:major\s+)?depression\b",
    r"\bi\s+(?:was|am|'m)\s+diagnosed\b.{0,80}\bdepress",
    r"\b(?:my|a)\s+(?:doctor|psychiatrist)\s+(?:has\s+)?diagnosed\s+me\b.{0,80}\bdepress"
]

EXCLUSION_PATTERNS = [
    r"\bmy\s+friend\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+sister\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+brother\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+mother\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+father\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+answer\s+to\s*:",
    r"\banswer\s+to\s*:",
    r"\bquestion\s*:",
    r"\bq\s*:"
]

anchor_patterns = [
    re.compile(
        p,
        re.IGNORECASE | re.DOTALL
    )
    for p in REFINED_ANCHOR_PATTERNS
]

exclusion_patterns = [
    re.compile(
        p,
        re.IGNORECASE | re.DOTALL
    )
    for p in EXCLUSION_PATTERNS
]


def is_refined_anchor_tweet(text):

    if text is None:
        return False

    text = str(text).strip()

    if not text:
        return False

    for pattern in exclusion_patterns:
        if pattern.search(text):
            return False

    for pattern in anchor_patterns:
        if pattern.search(text):
            return True

    return False


train_positive_users = set(
    train_df.loc[
        train_df["label"] == 1,
        "username"
    ].astype(str)
)

val_positive_users = set(
    val_df.loc[
        val_df["label"] == 1,
        "username"
    ].astype(str)
)

test_positive_users = set(
    test_df.loc[
        test_df["label"] == 1,
        "username"
    ].astype(str)
)

all_positive_users = (
    train_positive_users
    | val_positive_users
    | test_positive_users
)

print()
print("=" * 60)
print("POSITIVE USERS BY SPLIT")
print("=" * 60)

print(
    "TRAIN:",
    len(train_positive_users)
)

print(
    "VALIDATION:",
    len(val_positive_users)
)

print(
    "TEST:",
    len(test_positive_users)
)

assert len(train_positive_users) == 2013
assert len(val_positive_users) == 252
assert len(test_positive_users) == 251
assert len(all_positive_users) == 2516


timeline_files_by_user = defaultdict(list)

for timeline_file in d1_timeline_files:

    filename = os.path.basename(
        timeline_file
    )

    username = filename.split(
        "created_at"
    )[0]

    timeline_files_by_user[
        username
    ].append(
        timeline_file
    )

assert len(timeline_files_by_user) == 2516


anchor_records = []
users_processed = set()
tweets_scanned = 0

for username in sorted(
    all_positive_users
):

    if username not in timeline_files_by_user:
        raise KeyError(
            f"Missing D1 timeline for user: {username}"
        )

    users_processed.add(username)

    for timeline_file in timeline_files_by_user[
        username
    ]:

        with open(
            timeline_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                record = json.loads(line)

                tweets_scanned += 1

                tweet_id = str(
                    record.get(
                        "id_str",
                        record.get(
                            "id",
                            ""
                        )
                    )
                )

                tweet_text = str(
                    record.get(
                        "text",
                        ""
                    )
                )

                if (
                    tweet_id
                    and
                    is_refined_anchor_tweet(
                        tweet_text
                    )
                ):

                    anchor_records.append({
                        "username": username,
                        "tweet_id": tweet_id,
                        "tweet_text": tweet_text
                    })


print()
print("=" * 60)
print("D1 ANCHOR DETECTION SUMMARY")
print("=" * 60)

print(
    "Positive users processed:",
    len(users_processed)
)

print(
    "Tweets scanned:",
    tweets_scanned
)

print(
    "Raw anchor records:",
    len(anchor_records)
)

assert len(users_processed) == 2516
assert tweets_scanned == 508806
assert len(anchor_records) == 1920


EXACT_ANCHOR_SET = set()

for record in anchor_records:

    EXACT_ANCHOR_SET.add(
        (
            record["username"],
            record["tweet_id"]
        )
    )

print(
    "Unique anchor pairs:",
    len(EXACT_ANCHOR_SET)
)

duplicate_anchor_occurrences = (
    len(anchor_records)
    -
    len(EXACT_ANCHOR_SET)
)

print(
    "Duplicate anchor occurrences:",
    duplicate_anchor_occurrences
)

assert len(EXACT_ANCHOR_SET) == 1847
assert duplicate_anchor_occurrences == 73


train_anchor_users = set(
    train_df.loc[
        train_df["label"] == 1,
        "username"
    ].astype(str)
)

val_anchor_users = set(
    val_df.loc[
        val_df["label"] == 1,
        "username"
    ].astype(str)
)

test_anchor_users = set(
    test_df.loc[
        test_df["label"] == 1,
        "username"
    ].astype(str)
)

train_unique_anchors = sum(
    1
    for username, tweet_id
    in EXACT_ANCHOR_SET
    if username in train_anchor_users
)

val_unique_anchors = sum(
    1
    for username, tweet_id
    in EXACT_ANCHOR_SET
    if username in val_anchor_users
)

test_unique_anchors = sum(
    1
    for username, tweet_id
    in EXACT_ANCHOR_SET
    if username in test_anchor_users
)

print()
print("=" * 60)
print("UNIQUE ANCHORS BY SPLIT")
print("=" * 60)

print(
    "TRAIN unique anchors:",
    train_unique_anchors
)

print(
    "VAL unique anchors:",
    val_unique_anchors
)

print(
    "TEST unique anchors:",
    test_unique_anchors
)

assert train_unique_anchors == 1476
assert val_unique_anchors == 190
assert test_unique_anchors == 181


anchor_output = []

for username, tweet_id in sorted(
    EXACT_ANCHOR_SET
):

    if username in train_anchor_users:
        split_name = "TRAIN"

    elif username in val_anchor_users:
        split_name = "VAL"

    elif username in test_anchor_users:
        split_name = "TEST"

    else:
        continue

    anchor_output.append({
        "username": username,
        "tweet_id": tweet_id,
        "split": split_name
    })


anchor_df = pd.DataFrame(
    anchor_output
)

assert len(anchor_df) == 1847

anchor_df.to_csv(
    os.path.join(
        RESULT_DIR,
        "d1_unique_anchor_pairs.csv"
    ),
    index=False
)

with open(
    os.path.join(
        RESULT_DIR,
        "d1_anchor_detection_summary.json"
    ),
    "w"
) as f:

    json.dump(
        {
            "d1_timeline_files": 2626,
            "positive_users_processed": 2516,
            "tweets_scanned": 508806,
            "raw_anchor_records": 1920,
            "unique_anchor_pairs": 1847,
            "duplicate_anchor_occurrences": 73,
            "train_unique_anchors": 1476,
            "val_unique_anchors": 190,
            "test_unique_anchors": 181
        },
        f,
        indent=2
    )

print()
print("=" * 60)
print("CELL 3 COMPLETE")
print("=" * 60)

print("✓ Exact D1 anchor detection completed")
print("✓ 1,920 raw anchor records detected")
print("✓ 1,847 unique anchor pairs identified")
print("✓ 73 duplicate occurrences handled")
print("✓ TRAIN unique anchors = 1,476")
print("✓ VAL unique anchors = 190")
print("✓ TEST unique anchors = 181")
print("✓ Anchor detection summary saved")

BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR DETECTION

D1 timeline files found: 2626

POSITIVE USERS BY SPLIT
TRAIN: 2013
VALIDATION: 252
TEST: 251

D1 ANCHOR DETECTION SUMMARY
Positive users processed: 2516
Tweets scanned: 508806
Raw anchor records: 1920
Unique anchor pairs: 1847
Duplicate anchor occurrences: 73

UNIQUE ANCHORS BY SPLIT
TRAIN unique anchors: 1476
VAL unique anchors: 190
TEST unique anchors: 181

CELL 3 COMPLETE
✓ Exact D1 anchor detection completed
✓ 1,920 raw anchor records detected
✓ 1,847 unique anchor pairs identified
✓ 73 duplicate occurrences handled
✓ TRAIN unique anchors = 1,476
✓ VAL unique anchors = 190
✓ TEST unique anchors = 181
✓ Anchor detection summary saved


In [4]:
# ============================================================
# CELL 3 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR DETECTION")
print("=" * 60)

import os
import re
import json
import glob
from collections import defaultdict

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "vedantnawghare35/"
    "depression-detection-mddl-final-pbl/"
    "Dataset_MDDL"
)

D1_TIMELINE_DIR = os.path.join(
    DATASET_ROOT,
    "labeled",
    "positive",
    "data",
    "timeline"
)

assert os.path.exists(D1_TIMELINE_DIR)

d1_timeline_files = glob.glob(
    os.path.join(D1_TIMELINE_DIR, "*.json")
)

print()
print(
    "D1 timeline files found:",
    len(d1_timeline_files)
)

assert len(d1_timeline_files) == 2626

REFINED_ANCHOR_PATTERNS = [
    r"\bi(?:'m| am)\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi\s+(?:was|were)\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi(?:'ve| have)\s+been\s+diagnosed\s+with\s+(?:major\s+)?depression\b",
    r"\bi(?:'m| am)\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi\s+(?:was|were)\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi(?:'ve| have)\s+been\s+diagnosed\s+with\s+(?:major\s+)?depressive\s+disorder\b",
    r"\bi\s+have\s+(?:major\s+)?depression\b",
    r"\bi\s+suffer\s+from\s+(?:major\s+)?depression\b",
    r"\bi\s+(?:was|am|'m)\s+diagnosed\b.{0,80}\bdepress",
    r"\b(?:my|a)\s+(?:doctor|psychiatrist)\s+(?:has\s+)?diagnosed\s+me\b.{0,80}\bdepress"
]

EXCLUSION_PATTERNS = [
    r"\bmy\s+friend\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+sister\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+brother\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+mother\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+father\s+(?:was|is|has\s+been)\s+diagnosed",
    r"\bmy\s+answer\s+to\s*:",
    r"\banswer\s+to\s*:",
    r"\bquestion\s*:",
    r"\bq\s*:"
]

anchor_patterns = [
    re.compile(
        p,
        re.IGNORECASE | re.DOTALL
    )
    for p in REFINED_ANCHOR_PATTERNS
]

exclusion_patterns = [
    re.compile(
        p,
        re.IGNORECASE | re.DOTALL
    )
    for p in EXCLUSION_PATTERNS
]


def is_refined_anchor_tweet(text):
    if text is None:
        return False

    text = str(text).strip()

    if not text:
        return False

    for pattern in exclusion_patterns:
        if pattern.search(text):
            return False

    for pattern in anchor_patterns:
        if pattern.search(text):
            return True

    return False


train_positive_users = set(
    train_df.loc[
        train_df["label"] == 1,
        "username"
    ].astype(str)
)

val_positive_users = set(
    val_df.loc[
        val_df["label"] == 1,
        "username"
    ].astype(str)
)

test_positive_users = set(
    test_df.loc[
        test_df["label"] == 1,
        "username"
    ].astype(str)
)

all_positive_users = (
    train_positive_users
    |
    val_positive_users
    |
    test_positive_users
)

print()
print("=" * 60)
print("POSITIVE USERS BY SPLIT")
print("=" * 60)

print("TRAIN:", len(train_positive_users))
print("VALIDATION:", len(val_positive_users))
print("TEST:", len(test_positive_users))

assert len(train_positive_users) == 2013
assert len(val_positive_users) == 252
assert len(test_positive_users) == 251
assert len(all_positive_users) == 2516


timeline_files_by_user = defaultdict(list)

for timeline_file in d1_timeline_files:

    filename = os.path.basename(
        timeline_file
    )

    username = filename.split(
        "created_at"
    )[0]

    timeline_files_by_user[
        username
    ].append(
        timeline_file
    )

assert len(timeline_files_by_user) == 2516


anchor_records = []
users_processed = set()
tweets_scanned = 0

for username in sorted(
    all_positive_users
):

    if username not in timeline_files_by_user:
        raise KeyError(
            f"Missing D1 timeline for user: {username}"
        )

    users_processed.add(username)

    for timeline_file in timeline_files_by_user[
        username
    ]:

        with open(
            timeline_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                record = json.loads(line)

                tweets_scanned += 1

                tweet_id = str(
                    record.get(
                        "id_str",
                        record.get(
                            "id",
                            ""
                        )
                    )
                )

                tweet_text = str(
                    record.get(
                        "text",
                        ""
                    )
                )

                if (
                    tweet_id
                    and
                    is_refined_anchor_tweet(
                        tweet_text
                    )
                ):

                    anchor_records.append({
                        "username": username,
                        "tweet_id": tweet_id,
                        "created_at":
                            record.get(
                                "created_at"
                            ),
                        "text": tweet_text
                    })


anchor_pairs = {
    (
        record["username"],
        record["tweet_id"]
    )
    for record in anchor_records
}

train_anchor_pairs = {
    pair
    for pair in anchor_pairs
    if pair[0] in train_positive_users
}

val_anchor_pairs = {
    pair
    for pair in anchor_pairs
    if pair[0] in val_positive_users
}

test_anchor_pairs = {
    pair
    for pair in anchor_pairs
    if pair[0] in test_positive_users
}


print()
print("=" * 60)
print("EXACT ANCHOR SUMMARY")
print("=" * 60)

print(
    "Positive users processed:",
    len(users_processed)
)

print(
    "Tweets scanned:",
    tweets_scanned
)

print(
    "Raw anchor records:",
    len(anchor_records)
)

print(
    "Unique anchor pairs:",
    len(anchor_pairs)
)

print()
print(
    "TRAIN unique anchors:",
    len(train_anchor_pairs)
)

print(
    "VAL unique anchors:",
    len(val_anchor_pairs)
)

print(
    "TEST unique anchors:",
    len(test_anchor_pairs)
)

assert len(users_processed) == 2516
assert tweets_scanned == 508806
assert len(anchor_records) == 1920
assert len(anchor_pairs) == 1847
assert len(train_anchor_pairs) == 1476
assert len(val_anchor_pairs) == 190
assert len(test_anchor_pairs) == 181


# Save anchor records
bigbird_emotion_anchor_df = pd.DataFrame(
    anchor_records
)

bigbird_emotion_anchor_df.to_csv(
    os.path.join(
        RESULT_DIR,
        "exact_anchor_records.csv"
    ),
    index=False
)

pd.DataFrame(
    list(anchor_pairs),
    columns=[
        "username",
        "tweet_id"
    ]
).to_csv(
    os.path.join(
        RESULT_DIR,
        "unique_anchor_pairs.csv"
    ),
    index=False
)


print()
print("=" * 60)
print("CELL 3 COMPLETE")
print("=" * 60)

print("✓ Exact anchor detector reproduced")
print("✓ 1,847 unique anchors confirmed")
print("✓ 1,920 anchor records confirmed")
print("✓ TRAIN = 1,476 unique anchors")
print("✓ VAL = 190 unique anchors")
print("✓ TEST = 181 unique anchors")
print("✓ No D2 users included")
print("✓ Anchor records saved")
print("✓ No timelines modified yet")

BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR DETECTION

D1 timeline files found: 2626

POSITIVE USERS BY SPLIT
TRAIN: 2013
VALIDATION: 252
TEST: 251

EXACT ANCHOR SUMMARY
Positive users processed: 2516
Tweets scanned: 508806
Raw anchor records: 1920
Unique anchor pairs: 1847

TRAIN unique anchors: 1476
VAL unique anchors: 190
TEST unique anchors: 181

CELL 3 COMPLETE
✓ Exact anchor detector reproduced
✓ 1,847 unique anchors confirmed
✓ 1,920 anchor records confirmed
✓ TRAIN = 1,476 unique anchors
✓ VAL = 190 unique anchors
✓ TEST = 181 unique anchors
✓ No D2 users included
✓ Anchor records saved
✓ No timelines modified yet


In [5]:
# ============================================================
# CELL 4 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR REMOVAL")
print("=" * 60)

EXACT_ANCHOR_SET = set(anchor_pairs)

train_anchor_records = [
    r for r in anchor_records
    if r["username"] in train_positive_users
]

val_anchor_records = [
    r for r in anchor_records
    if r["username"] in val_positive_users
]

test_anchor_records = [
    r for r in anchor_records
    if r["username"] in test_positive_users
]

print()
print("=" * 60)
print("PRE-REMOVAL ANCHOR VERIFICATION")
print("=" * 60)

duplicate_anchor_occurrences = (
    len(anchor_records) - len(anchor_pairs)
)

print("Raw anchor records:", len(anchor_records))
print("Unique anchor pairs:", len(anchor_pairs))
print(
    "Duplicate anchor occurrences:",
    duplicate_anchor_occurrences
)

assert len(anchor_records) == 1920
assert len(anchor_pairs) == 1847
assert duplicate_anchor_occurrences == 73

print("✓ 1,847 unique anchors verified")
print("✓ 73 duplicate occurrences verified")
print("✓ 1,920 total anchor records verified")

print()
print("=" * 60)
print("SPLIT-LEVEL ANCHOR RECORD COUNTS")
print("=" * 60)

print(
    "TRAIN:",
    len(train_anchor_records),
    "| expected: 1535"
)

print(
    "VAL:",
    len(val_anchor_records),
    "| expected: 199"
)

print(
    "TEST:",
    len(test_anchor_records),
    "| expected: 186"
)

assert len(train_anchor_records) == 1535
assert len(val_anchor_records) == 199
assert len(test_anchor_records) == 186


# ============================================================
# TIMELINE INDEX
# ============================================================

timeline_files_by_user = defaultdict(list)

for timeline_file in d1_timeline_files:

    filename = os.path.basename(
        timeline_file
    )

    username = filename.split(
        "created_at"
    )[0]

    timeline_files_by_user[
        username
    ].append(
        timeline_file
    )

assert len(timeline_files_by_user) == 2516


def build_anchor_removed_text(username):

    kept_texts = []
    removed_count = 0

    for timeline_file in timeline_files_by_user[
        username
    ]:

        with open(
            timeline_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                record = json.loads(line)

                tweet_id = str(
                    record.get(
                        "id_str",
                        record.get("id", "")
                    )
                )

                if (
                    str(username),
                    tweet_id
                ) in EXACT_ANCHOR_SET:

                    removed_count += 1
                    continue

                tweet_text = str(
                    record.get(
                        "text",
                        ""
                    )
                ).strip()

                if tweet_text:
                    kept_texts.append(
                        tweet_text
                    )

    return (
        " ".join(kept_texts),
        removed_count
    )


# ============================================================
# COPY SOURCE DATA
# ============================================================

bigbird_emotion_df_train = (
    train_df.copy(deep=True)
)

bigbird_emotion_df_val = (
    val_df.copy(deep=True)
)

bigbird_emotion_df_test = (
    test_df.copy(deep=True)
)

positive_anchor_removed_text = {}

train_removed = 0
val_removed = 0
test_removed = 0

for username in sorted(
    all_positive_users
):

    text, removed = (
        build_anchor_removed_text(
            username
        )
    )

    positive_anchor_removed_text[
        username
    ] = text

    if username in train_positive_users:

        train_removed += removed

    elif username in val_positive_users:

        val_removed += removed

    elif username in test_positive_users:

        test_removed += removed


print()
print("=" * 60)
print("FINAL ANCHOR REMOVAL COUNTS")
print("=" * 60)

print(
    "TRAIN removed:",
    train_removed
)

print(
    "VAL removed:",
    val_removed
)

print(
    "TEST removed:",
    test_removed
)

print(
    "TOTAL removed:",
    (
        train_removed
        +
        val_removed
        +
        test_removed
    )
)

assert train_removed == 1535
assert val_removed == 199
assert test_removed == 186

assert (
    train_removed
    +
    val_removed
    +
    test_removed
) == 1920

print(
    "✓ Record-level removal exactly "
    "matches previous experiments"
)


# ============================================================
# ASSIGN ANCHOR-REMOVED TEXT
# ============================================================

for df in [
    bigbird_emotion_df_train,
    bigbird_emotion_df_val,
    bigbird_emotion_df_test
]:

    positive_mask = (
        df["label"] == 1
    )

    df.loc[
        positive_mask,
        "anchor_removed_text"
    ] = (
        df.loc[
            positive_mask,
            "username"
        ]
        .astype(str)
        .map(
            positive_anchor_removed_text
        )
        .fillna("")
        .values
    )

    negative_mask = (
        df["label"] == 0
    )

    df.loc[
        negative_mask,
        "anchor_removed_text"
    ] = (
        df.loc[
            negative_mask,
            "timeline_text"
        ]
        .fillna("")
        .astype(str)
        .values
    )


# ============================================================
# D2 PRESERVATION CHECK
# ============================================================

print()
print("=" * 60)
print("D2 PRESERVATION CHECK")
print("=" * 60)

for original_df, result_df, name in [

    (
        train_df,
        bigbird_emotion_df_train,
        "TRAIN"
    ),

    (
        val_df,
        bigbird_emotion_df_val,
        "VALIDATION"
    ),

    (
        test_df,
        bigbird_emotion_df_test,
        "TEST"
    )
]:

    original_d2 = (
        original_df.loc[
            original_df["label"] == 0,
            [
                "username",
                "timeline_text"
            ]
        ]
        .set_index("username")
        .sort_index()
    )

    result_d2 = (
        result_df.loc[
            result_df["label"] == 0,
            [
                "username",
                "anchor_removed_text"
            ]
        ]
        .rename(
            columns={
                "anchor_removed_text":
                    "timeline_text"
            }
        )
        .set_index("username")
        .sort_index()
    )

    assert original_d2.equals(
        result_d2
    )

    print(
        f"✓ {name} D2 unchanged"
    )


# ============================================================
# USER + LABEL PRESERVATION
# ============================================================

print()
print("=" * 60)
print("USER + LABEL PRESERVATION")
print("=" * 60)

for original_df, result_df, name in [

    (
        train_df,
        bigbird_emotion_df_train,
        "TRAIN"
    ),

    (
        val_df,
        bigbird_emotion_df_val,
        "VALIDATION"
    ),

    (
        test_df,
        bigbird_emotion_df_test,
        "TEST"
    )
]:

    assert (
        original_df["user_id"].tolist()
        ==
        result_df["user_id"].tolist()
    )

    assert (
        original_df["username"].tolist()
        ==
        result_df["username"].tolist()
    )

    assert (
        original_df["label"].tolist()
        ==
        result_df["label"].tolist()
    )

    print(
        f"✓ {name}: users and labels unchanged"
    )


# ============================================================
# ANCHOR-REMOVED TEXT INTEGRITY
# ============================================================

print()
print("=" * 60)
print("ANCHOR-REMOVED TEXT INTEGRITY")
print("=" * 60)

for name, df in [

    (
        "TRAIN",
        bigbird_emotion_df_train
    ),

    (
        "VALIDATION",
        bigbird_emotion_df_val
    ),

    (
        "TEST",
        bigbird_emotion_df_test
    )
]:

    empty_count = int(
        (
            df["anchor_removed_text"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
        ).sum()
    )

    print(
        f"{name}: empty anchor-removed texts = "
        f"{empty_count}"
    )


# ============================================================
# SAVE
# ============================================================

bigbird_emotion_df_train.to_pickle(
    os.path.join(
        RESULT_DIR,
        "train_anchor_removed.pkl"
    )
)

bigbird_emotion_df_val.to_pickle(
    os.path.join(
        RESULT_DIR,
        "val_anchor_removed.pkl"
    )
)

bigbird_emotion_df_test.to_pickle(
    os.path.join(
        RESULT_DIR,
        "test_anchor_removed.pkl"
    )
)


print()
print("=" * 60)
print("CELL 4 COMPLETE")
print("=" * 60)

print("✓ 1,847 unique D1 anchors removed")
print("✓ 73 duplicate occurrences handled")
print("✓ 1,920 total anchor records removed")
print("✓ TRAIN = 1,535 removed")
print("✓ VAL = 199 removed")
print("✓ TEST = 186 removed")
print("✓ D2 completely preserved")
print("✓ User IDs preserved")
print("✓ Labels preserved")
print("✓ Anchor-removed source saved")

BIGBIRD + EMOTION-BASED — EXACT D1 ANCHOR REMOVAL

PRE-REMOVAL ANCHOR VERIFICATION
Raw anchor records: 1920
Unique anchor pairs: 1847
Duplicate anchor occurrences: 73
✓ 1,847 unique anchors verified
✓ 73 duplicate occurrences verified
✓ 1,920 total anchor records verified

SPLIT-LEVEL ANCHOR RECORD COUNTS
TRAIN: 1535 | expected: 1535
VAL: 199 | expected: 199
TEST: 186 | expected: 186

FINAL ANCHOR REMOVAL COUNTS
TRAIN removed: 1535
VAL removed: 199
TEST removed: 186
TOTAL removed: 1920
✓ Record-level removal exactly matches previous experiments

D2 PRESERVATION CHECK
✓ TRAIN D2 unchanged
✓ VALIDATION D2 unchanged
✓ TEST D2 unchanged

USER + LABEL PRESERVATION
✓ TRAIN: users and labels unchanged
✓ VALIDATION: users and labels unchanged
✓ TEST: users and labels unchanged

ANCHOR-REMOVED TEXT INTEGRITY
TRAIN: empty anchor-removed texts = 29
VALIDATION: empty anchor-removed texts = 4
TEST: empty anchor-removed texts = 1

CELL 4 COMPLETE
✓ 1,847 unique D1 anchors removed
✓ 73 duplicate occu

In [ ]:
# ============================================================
# CELL 5 — BIGBIRD + EMOTION-BASED
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from kaggle_secrets import UserSecretsClient
import torch.nn.functional as F
from tqdm.auto import tqdm

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — EMOTION EXTRACTION")
print("=" * 60)

EMOTION_MODEL_NAME = (
    "cardiffnlp/twitter-roberta-base-emotion-latest"
)

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

print()
print("Emotion model:", EMOTION_MODEL_NAME)
print("Emotion features:", len(EMOTION_LABELS))
print("✓ HF_TOKEN retrieved")

emotion_tokenizer = AutoTokenizer.from_pretrained(
    EMOTION_MODEL_NAME,
    token=hf_token
)

emotion_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        EMOTION_MODEL_NAME,
        token=hf_token
    )
    .to(device)
)

emotion_model.eval()

for p in emotion_model.parameters():
    p.requires_grad = False

print("✓ Emotion tokenizer loaded")
print("✓ Emotion model loaded")
print("✓ Emotion model frozen")

model_labels = [
    emotion_model.config.id2label[i]
    for i in range(
        emotion_model.config.num_labels
    )
]

print()
print("Model emotion labels:")
print(model_labels)

assert emotion_model.config.num_labels == 11
assert model_labels == EMOTION_LABELS


def extract_emotion_features(
    texts,
    split_name,
    batch_size=32
):

    features = []
    empty_count = 0
    nonempty_indices = []

    for i, text in enumerate(texts):

        if str(text).strip() == "":
            empty_count += 1
        else:
            nonempty_indices.append(i)

    output = np.zeros(
        (len(texts), 11),
        dtype=np.float32
    )

    print()
    print(f"Processing {split_name}...")
    print("Users:", len(texts))
    print("Empty texts:", empty_count)
    print("Non-empty texts:", len(nonempty_indices))

    total_batches = (
        len(nonempty_indices) + batch_size - 1
    ) // batch_size

    with torch.no_grad():

        for start in tqdm(
            range(
                0,
                len(nonempty_indices),
                batch_size
            ),
            desc=split_name
        ):

            batch_indices = nonempty_indices[
                start:start + batch_size
            ]

            batch_texts = [
                str(texts[i])
                for i in batch_indices
            ]

            encoded = emotion_tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            )

            encoded = {
                k: v.to(device)
                for k, v in encoded.items()
            }

            logits = emotion_model(
                **encoded
            ).logits.float()

            probabilities = torch.sigmoid(
                logits
            )

            output[
                batch_indices
            ] = probabilities.cpu().numpy()

    assert output.shape == (
        len(texts),
        11
    )

    assert np.isfinite(output).all()
    assert output.min() >= 0.0
    assert output.max() <= 1.0

    return output


train_emotion_features = extract_emotion_features(
    bigbird_emotion_df_train[
        "anchor_removed_text"
    ].fillna("").tolist(),
    "TRAIN"
)

val_emotion_features = extract_emotion_features(
    bigbird_emotion_df_val[
        "anchor_removed_text"
    ].fillna("").tolist(),
    "VALIDATION"
)

test_emotion_features = extract_emotion_features(
    bigbird_emotion_df_test[
        "anchor_removed_text"
    ].fillna("").tolist(),
    "TEST"
)

print()
print("=" * 60)
print("EMOTION FEATURE SHAPES")
print("=" * 60)

print(
    "Train:",
    train_emotion_features.shape
)

print(
    "Validation:",
    val_emotion_features.shape
)

print(
    "Test:",
    test_emotion_features.shape
)

assert train_emotion_features.shape == (
    6135,
    11
)

assert val_emotion_features.shape == (
    767,
    11
)

assert test_emotion_features.shape == (
    767,
    11
)

print("✓ Exact 11 emotion features confirmed")

print()
print("=" * 60)
print("EMOTION FEATURE INTEGRITY")
print("=" * 60)

for features, name in [
    (
        train_emotion_features,
        "TRAIN"
    ),
    (
        val_emotion_features,
        "VALIDATION"
    ),
    (
        test_emotion_features,
        "TEST"
    )
]:

    print(
        f"{name}: "
        f"NaN={np.isnan(features).sum()}, "
        f"Inf={np.isinf(features).sum()}"
    )

    print(
        f"{name}: "
        f"min={features.min():.6f}, "
        f"max={features.max():.6f}, "
        f"mean={features.mean():.6f}"
    )

    assert np.isnan(features).sum() == 0
    assert np.isinf(features).sum() == 0

emotion_columns = [
    f"emotion_{label}"
    for label in EMOTION_LABELS
]

print()
print("=" * 60)
print("EMOTION FEATURE COLUMNS")
print("=" * 60)

for i, column in enumerate(
    emotion_columns,
    start=1
):

    print(
        f"{i:02d}. {column}"
    )

# Save raw emotion features
np.save(
    os.path.join(
        RESULT_DIR,
        "train_emotion_features.npy"
    ),
    train_emotion_features
)

np.save(
    os.path.join(
        RESULT_DIR,
        "val_emotion_features.npy"
    ),
    val_emotion_features
)

np.save(
    os.path.join(
        RESULT_DIR,
        "test_emotion_features.npy"
    ),
    test_emotion_features
)

pd.DataFrame(
    train_emotion_features,
    columns=emotion_columns
).to_csv(
    os.path.join(
        RESULT_DIR,
        "train_emotion_features.csv"
    ),
    index=False
)

pd.DataFrame(
    val_emotion_features,
    columns=emotion_columns
).to_csv(
    os.path.join(
        RESULT_DIR,
        "val_emotion_features.csv"
    ),
    index=False
)

pd.DataFrame(
    test_emotion_features,
    columns=emotion_columns
).to_csv(
    os.path.join(
        RESULT_DIR,
        "test_emotion_features.csv"
    ),
    index=False
)

print()
print("=" * 60)
print("CELL 5 COMPLETE")
print("=" * 60)

print("✓ Emotion model applied to anchor-removed text")
print("✓ Multilabel sigmoid probabilities used")
print("✓ 11 emotion features extracted")
print("✓ Train = 6135 × 11")
print("✓ Validation = 767 × 11")
print("✓ Test = 767 × 11")
print("✓ No NaN / Inf")
print("✓ Emotion features saved")

BIGBIRD + EMOTION-BASED — EMOTION EXTRACTION


In [ ]:
# ============================================================
# CELL 6A — BIGBIRD + EMOTION-BASED
# EXACT 53-FEATURE METADATA EXTRACTION + NORMALIZATION
# ============================================================

from sklearn.preprocessing import StandardScaler

behavioral_features = [
    "tweet_count", "posting_frequency", "active_days",
    "night_posting_ratio", "engagement", "total_retweets",
    "total_favorites", "follower_friend_ratio"
]

linguistic_features = [
    "lexical_diversity", "word_count",
    "avg_paragraph_length", "repeated_letters"
]

readability_features = [
    "flesch_reading_ease", "gunning_fog",
    "flesch_kincaid_grade"
]

emotional_features = [
    "sentiment_score", "emotion_score",
    "pos_emoji_count", "neg_emoji_count",
    "emoji_density", "exclamation_count"
]

psychological_features = [
    "depression_lexicon_score", "hopelessness_score",
    "loneliness_score", "anxiety_score",
    "sleep_words_ratio", "fatigue_words_ratio",
    "selfharm_keywords_ratio"
]

writing_style_features = [
    "first_person_ratio", "negation_ratio",
    "uppercase_ratio", "punctuation_freq",
    "avg_sentence_length"
]

non_lda_features = (
    behavioral_features
    + linguistic_features
    + readability_features
    + emotional_features
    + psychological_features
    + writing_style_features
)

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — METADATA EXTRACTION + NORMALIZATION")
print("=" * 60)

print()
print("Behavioral:", len(behavioral_features))
print("Linguistic:", len(linguistic_features))
print("Readability:", len(readability_features))
print("Emotional:", len(emotional_features))
print("Psychological:", len(psychological_features))
print("Writing Style:", len(writing_style_features))
print("LDA: 20")
print()
print("TOTAL EXPECTED: 53")

assert len(non_lda_features) == 33


# ============================================================
# NON-LDA FEATURE VERIFICATION
# ============================================================

print()
print("=" * 60)
print("NON-LDA FEATURE VERIFICATION")
print("=" * 60)

for df, name in [
    (bigbird_emotion_df_train, "TRAIN"),
    (bigbird_emotion_df_val, "VALIDATION"),
    (bigbird_emotion_df_test, "TEST")
]:

    missing = [
        c for c in non_lda_features
        if c not in df.columns
    ]

    assert not missing

    print(
        f"✓ {name}: all 33 non-LDA features found"
    )


# ============================================================
# PHYSICAL SCHEMA
# ============================================================

train_columns = list(
    bigbird_emotion_df_train.columns
)

val_columns = list(
    bigbird_emotion_df_val.columns
)

test_columns = list(
    bigbird_emotion_df_test.columns
)

assert len(train_columns) == 83
assert len(val_columns) == 83
assert len(test_columns) == 83

assert (
    train_columns
    ==
    val_columns
    ==
    test_columns
)

print()
print("=" * 60)
print("PHYSICAL SCHEMA")
print("=" * 60)

print("Train columns:", len(train_columns))
print("Validation columns:", len(val_columns))
print("Test columns:", len(test_columns))
print("✓ 83-column current schema confirmed")


# ============================================================
# NON-LDA POSITIONS
# ============================================================

non_lda_positions = []

for feature in non_lda_features:

    positions = [
        i
        for i, col in enumerate(train_columns)
        if col == feature
    ]

    assert len(positions) == 1

    non_lda_positions.append(
        positions[0]
    )

print()
print("Non-LDA physical positions:")
print(non_lda_positions)


# ============================================================
# LDA POSITIONS
# ============================================================

first_lda_positions = list(
    range(42, 62)
)

print()
print("=" * 60)
print("LDA COLUMN SELECTION")
print("=" * 60)

for i in range(20):

    p = 42 + i

    assert train_columns[p] == (
        f"lda_topic_{i + 1}"
    )

    print(
        f"lda_topic_{i + 1}: "
        f"using position {p}"
    )

print()
print("✓ First LDA occurrence selected")


# ============================================================
# FINAL METADATA SELECTION
# ============================================================

selected_metadata_positions = (
    non_lda_positions
    + first_lda_positions
)

assert len(
    selected_metadata_positions
) == 53

assert len(
    set(selected_metadata_positions)
) == 53

selected_metadata_names = (
    non_lda_features
    + [
        train_columns[i]
        for i in first_lda_positions
    ]
)

print()
print("=" * 60)
print("FINAL METADATA SELECTION")
print("=" * 60)

print(
    "Non-LDA features:",
    len(non_lda_features)
)

print(
    "LDA features:",
    len(first_lda_positions)
)

print(
    "Total selected:",
    len(selected_metadata_positions)
)


# ============================================================
# EXTRACTION
# ============================================================

def extract_metadata(df):
    return df.iloc[
        :,
        selected_metadata_positions
    ].to_numpy(
        dtype=np.float32
    )


print()
print("Extracting metadata — TRAIN")

train_bigbird_emotion_metadata = (
    extract_metadata(
        bigbird_emotion_df_train
    )
)

print(
    "TRAIN metadata:",
    train_bigbird_emotion_metadata.shape
)

print()
print("Extracting metadata — VALIDATION")

val_bigbird_emotion_metadata = (
    extract_metadata(
        bigbird_emotion_df_val
    )
)

print(
    "VALIDATION metadata:",
    val_bigbird_emotion_metadata.shape
)

print()
print("Extracting metadata — TEST")

test_bigbird_emotion_metadata = (
    extract_metadata(
        bigbird_emotion_df_test
    )
)

print(
    "TEST metadata:",
    test_bigbird_emotion_metadata.shape
)


# ============================================================
# SHAPE VERIFICATION
# ============================================================

print()
print("=" * 60)
print("METADATA SHAPES")
print("=" * 60)

print(
    "Train:",
    train_bigbird_emotion_metadata.shape
)

print(
    "Validation:",
    val_bigbird_emotion_metadata.shape
)

print(
    "Test:",
    test_bigbird_emotion_metadata.shape
)

assert (
    train_bigbird_emotion_metadata.shape
    ==
    (6135, 53)
)

assert (
    val_bigbird_emotion_metadata.shape
    ==
    (767, 53)
)

assert (
    test_bigbird_emotion_metadata.shape
    ==
    (767, 53)
)

print("✓ Exact 53 metadata features confirmed")


# ============================================================
# NUMERICAL INTEGRITY
# ============================================================

print()
print("=" * 60)
print("METADATA NUMERICAL INTEGRITY")
print("=" * 60)

for metadata, name in [
    (
        train_bigbird_emotion_metadata,
        "TRAIN"
    ),
    (
        val_bigbird_emotion_metadata,
        "VALIDATION"
    ),
    (
        test_bigbird_emotion_metadata,
        "TEST"
    )
]:

    nan_count = int(
        np.isnan(metadata).sum()
    )

    inf_count = int(
        np.isinf(metadata).sum()
    )

    print(
        f"{name}: "
        f"NaN={nan_count}, "
        f"Inf={inf_count}"
    )

    assert nan_count == 0
    assert inf_count == 0


# ============================================================
# TRAIN-ONLY NORMALIZATION
# ============================================================

print()
print("=" * 60)
print("TRAIN-ONLY METADATA NORMALIZATION")
print("=" * 60)

metadata_scaler = StandardScaler()

train_bigbird_emotion_metadata_scaled = (
    metadata_scaler
    .fit_transform(
        train_bigbird_emotion_metadata
    )
    .astype(np.float32)
)

val_bigbird_emotion_metadata_scaled = (
    metadata_scaler
    .transform(
        val_bigbird_emotion_metadata
    )
    .astype(np.float32)
)

test_bigbird_emotion_metadata_scaled = (
    metadata_scaler
    .transform(
        test_bigbird_emotion_metadata
    )
    .astype(np.float32)
)

training_mean_abs = float(
    np.abs(
        train_bigbird_emotion_metadata_scaled
        .mean(axis=0)
    ).mean()
)

training_std_mean = float(
    train_bigbird_emotion_metadata_scaled
    .std(axis=0)
    .mean()
)

print(
    "Training mean absolute mean:",
    training_mean_abs
)

print(
    "Training mean std:",
    training_std_mean
)

print()
print("Scaler fitted on: TRAIN ONLY")
print("Validation: transform only")
print("Test: transform only")

assert training_mean_abs < 1e-5
assert abs(training_std_mean - 1.0) < 1e-5


# ============================================================
# SAVE
# ============================================================

np.save(
    os.path.join(
        RESULT_DIR,
        "train_metadata_scaled.npy"
    ),
    train_bigbird_emotion_metadata_scaled
)

np.save(
    os.path.join(
        RESULT_DIR,
        "val_metadata_scaled.npy"
    ),
    val_bigbird_emotion_metadata_scaled
)

np.save(
    os.path.join(
        RESULT_DIR,
        "test_metadata_scaled.npy"
    ),
    test_bigbird_emotion_metadata_scaled
)

with open(
    os.path.join(
        RESULT_DIR,
        "metadata_columns.json"
    ),
    "w"
) as f:
    json.dump(
        selected_metadata_names,
        f,
        indent=2
    )

with open(
    os.path.join(
        RESULT_DIR,
        "metadata_physical_positions.json"
    ),
    "w"
) as f:
    json.dump(
        selected_metadata_positions,
        f,
        indent=2
    )

print()
print("=" * 60)
print("CELL 6A COMPLETE")
print("=" * 60)

print("✓ 33 non-LDA metadata features selected")
print("✓ First 20 LDA features selected")
print("✓ Exact 53 metadata features confirmed")
print("✓ Train = 6135 × 53")
print("✓ Validation = 767 × 53")
print("✓ Test = 767 × 53")
print("✓ No NaN / Inf")
print("✓ StandardScaler fitted on TRAIN only")
print("✓ Validation transformed only")
print("✓ Test transformed only")
print("✓ Metadata arrays saved")
print("✓ Column mapping saved")

In [ ]:
# ============================================================
# CELL 6B — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — EMOTION NORMALIZATION")
print("=" * 60)

print()
print("Emotion features:", train_emotion_features.shape[1])
print("Normalization: TRAIN ONLY")

# Emotion scaler
emotion_scaler = StandardScaler()

train_emotion_scaled = (
    emotion_scaler
    .fit_transform(
        train_emotion_features
    )
    .astype(np.float32)
)

val_emotion_scaled = (
    emotion_scaler
    .transform(
        val_emotion_features
    )
    .astype(np.float32)
)

test_emotion_scaled = (
    emotion_scaler
    .transform(
        test_emotion_features
    )
    .astype(np.float32)
)

print()
print("=" * 60)
print("NORMALIZED EMOTION SHAPES")
print("=" * 60)

print(
    "Train:",
    train_emotion_scaled.shape
)

print(
    "Validation:",
    val_emotion_scaled.shape
)

print(
    "Test:",
    test_emotion_scaled.shape
)

assert train_emotion_scaled.shape == (
    6135,
    11
)

assert val_emotion_scaled.shape == (
    767,
    11
)

assert test_emotion_scaled.shape == (
    767,
    11
)

print("✓ Emotion shapes verified")

print()
print("=" * 60)
print("NORMALIZATION CHECK")
print("=" * 60)

train_mean = np.abs(
    train_emotion_scaled.mean(axis=0)
).mean()

train_std = (
    train_emotion_scaled.std(axis=0)
).mean()

print(
    "Training mean absolute mean:",
    float(train_mean)
)

print(
    "Training mean std:",
    float(train_std)
)

assert train_mean < 1e-5
assert abs(train_std - 1.0) < 1e-5

print("✓ TRAIN emotion scaler verified")
print("✓ Validation transformed only")
print("✓ Test transformed only")

print()
print("=" * 60)
print("EMOTION + METADATA INPUT CHECK")
print("=" * 60)

print(
    "Metadata features:",
    train_bigbird_emotion_metadata_scaled.shape
)

print(
    "Emotion features:",
    train_emotion_scaled.shape
)

assert train_bigbird_emotion_metadata_scaled.shape == (
    6135,
    53
)

assert val_bigbird_emotion_metadata_scaled.shape == (
    767,
    53
)

assert test_bigbird_emotion_metadata_scaled.shape == (
    767,
    53
)

assert train_emotion_scaled.shape == (
    6135,
    11
)

assert val_emotion_scaled.shape == (
    767,
    11
)

assert test_emotion_scaled.shape == (
    767,
    11
)

print("✓ Metadata branch = 53 features")
print("✓ Emotion branch = 11 features")

# Save normalized emotion features
np.save(
    os.path.join(
        RESULT_DIR,
        "train_emotion_scaled.npy"
    ),
    train_emotion_scaled
)

np.save(
    os.path.join(
        RESULT_DIR,
        "val_emotion_scaled.npy"
    ),
    val_emotion_scaled
)

np.save(
    os.path.join(
        RESULT_DIR,
        "test_emotion_scaled.npy"
    ),
    test_emotion_scaled
)

emotion_scaler_info = {
    "scaler": "StandardScaler",
    "fit_on": "TRAIN ONLY",
    "features": 11,
    "emotion_model": EMOTION_MODEL_NAME,
    "labels": EMOTION_LABELS
}

with open(
    os.path.join(
        RESULT_DIR,
        "emotion_scaler_config.json"
    ),
    "w"
) as f:
    json.dump(
        emotion_scaler_info,
        f,
        indent=2
    )

print()
print("=" * 60)
print("CELL 6B COMPLETE")
print("=" * 60)

print("✓ 11 emotion features normalized")
print("✓ StandardScaler fitted on TRAIN only")
print("✓ Validation transformed only")
print("✓ Test transformed only")
print("✓ Emotion arrays saved")
print("✓ Metadata + emotion inputs ready")

In [ ]:
# ============================================================
# CELL 7A — BIGBIRD + EMOTION-BASED
# ============================================================

from transformers import AutoTokenizer
from tqdm.auto import tqdm

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — TOKENIZATION")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token
)

print()
print("Tokenizer:", MODEL_NAME)
print("Max length:", MAX_LENGTH)


def tokenize_texts(
    texts,
    split_name,
    batch_size=32
):

    all_input_ids = []
    all_attention_masks = []

    print()
    print(f"Processing {split_name}...")
    print("Users:", len(texts))

    for start in tqdm(
        range(
            0,
            len(texts),
            batch_size
        ),
        desc=split_name
    ):

        batch_texts = [
            str(text)
            for text in texts[
                start:start + batch_size
            ]
        ]

        encoded = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_attention_mask=True
        )

        all_input_ids.extend(
            encoded["input_ids"]
        )

        all_attention_masks.extend(
            encoded["attention_mask"]
        )

    input_ids = np.asarray(
        all_input_ids,
        dtype=np.int64
    )

    attention_masks = np.asarray(
        all_attention_masks,
        dtype=np.int64
    )

    return input_ids, attention_masks


train_input_ids, train_attention_masks = (
    tokenize_texts(
        bigbird_emotion_df_train[
            "anchor_removed_text"
        ].fillna("").tolist(),
        "TRAIN"
    )
)

val_input_ids, val_attention_masks = (
    tokenize_texts(
        bigbird_emotion_df_val[
            "anchor_removed_text"
        ].fillna("").tolist(),
        "VALIDATION"
    )
)

test_input_ids, test_attention_masks = (
    tokenize_texts(
        bigbird_emotion_df_test[
            "anchor_removed_text"
        ].fillna("").tolist(),
        "TEST"
    )
)

print()
print("=" * 60)
print("TOKENIZATION SHAPES")
print("=" * 60)

print(
    "TRAIN input IDs:",
    train_input_ids.shape
)

print(
    "TRAIN attention masks:",
    train_attention_masks.shape
)

print(
    "VALIDATION input IDs:",
    val_input_ids.shape
)

print(
    "VALIDATION attention masks:",
    val_attention_masks.shape
)

print(
    "TEST input IDs:",
    test_input_ids.shape
)

print(
    "TEST attention masks:",
    test_attention_masks.shape
)

assert train_input_ids.shape == (
    6135,
    512
)

assert train_attention_masks.shape == (
    6135,
    512
)

assert val_input_ids.shape == (
    767,
    512
)

assert val_attention_masks.shape == (
    767,
    512
)

assert test_input_ids.shape == (
    767,
    512
)

assert test_attention_masks.shape == (
    767,
    512
)

print()
print("=" * 60)
print("TOKEN VALUE INTEGRITY")
print("=" * 60)

for array, name in [
    (train_input_ids, "TRAIN input IDs"),
    (train_attention_masks, "TRAIN attention masks"),
    (val_input_ids, "VAL input IDs"),
    (val_attention_masks, "VAL attention masks"),
    (test_input_ids, "TEST input IDs"),
    (test_attention_masks, "TEST attention masks")
]:

    assert np.isfinite(array).all()

    print(
        f"✓ {name}: finite values only"
    )

print()
print("=" * 60)
print("ACTIVE TOKEN CHECK")
print("=" * 60)

print(
    "TRAIN mean active tokens:",
    float(
        train_attention_masks.sum(axis=1).mean()
    )
)

print(
    "VALIDATION mean active tokens:",
    float(
        val_attention_masks.sum(axis=1).mean()
    )
)

print(
    "TEST mean active tokens:",
    float(
        test_attention_masks.sum(axis=1).mean()
    )
)

print()
print("=" * 60)
print("CELL 7A COMPLETE")
print("=" * 60)

print("✓ BigBird tokenizer loaded")
print("✓ Max length = 512")
print("✓ Truncation enabled")
print("✓ Padding = max_length")
print("✓ TRAIN token inputs created")
print("✓ VALIDATION token inputs created")
print("✓ TEST token inputs created")
print("✓ All token shapes verified")

In [ ]:
# ============================================================
# CELL 7 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — FINAL INPUT ALIGNMENT")
print("=" * 60)

train_labels = (
    bigbird_emotion_df_train["label"]
    .to_numpy(dtype=np.int64)
)

val_labels = (
    bigbird_emotion_df_val["label"]
    .to_numpy(dtype=np.int64)
)

test_labels = (
    bigbird_emotion_df_test["label"]
    .to_numpy(dtype=np.int64)
)

print()
print("=" * 60)
print("INPUT SHAPES")
print("=" * 60)

print("TRAIN tokens:", train_input_ids.shape)
print(
    "TRAIN metadata:",
    train_bigbird_emotion_metadata_scaled.shape
)
print("TRAIN emotions:", train_emotion_scaled.shape)
print("TRAIN labels:", train_labels.shape)

print()
print("VALIDATION tokens:", val_input_ids.shape)
print(
    "VALIDATION metadata:",
    val_bigbird_emotion_metadata_scaled.shape
)
print("VALIDATION emotions:", val_emotion_scaled.shape)
print("VALIDATION labels:", val_labels.shape)

print()
print("TEST tokens:", test_input_ids.shape)
print(
    "TEST metadata:",
    test_bigbird_emotion_metadata_scaled.shape
)
print("TEST emotions:", test_emotion_scaled.shape)
print("TEST labels:", test_labels.shape)

assert train_input_ids.shape == (6135, 512)
assert train_attention_masks.shape == (6135, 512)
assert train_bigbird_emotion_metadata_scaled.shape == (
    6135, 53
)
assert train_emotion_scaled.shape == (6135, 11)
assert train_labels.shape == (6135,)

assert val_input_ids.shape == (767, 512)
assert val_attention_masks.shape == (767, 512)
assert val_bigbird_emotion_metadata_scaled.shape == (
    767, 53
)
assert val_emotion_scaled.shape == (767, 11)
assert val_labels.shape == (767,)

assert test_input_ids.shape == (767, 512)
assert test_attention_masks.shape == (767, 512)
assert test_bigbird_emotion_metadata_scaled.shape == (
    767, 53
)
assert test_emotion_scaled.shape == (767, 11)
assert test_labels.shape == (767,)

print()
print("✓ All input shapes correct")

print()
print("=" * 60)
print("USER + LABEL ALIGNMENT")
print("=" * 60)

assert len(bigbird_emotion_df_train) == len(
    train_emotion_scaled
)

assert len(bigbird_emotion_df_val) == len(
    val_emotion_scaled
)

assert len(bigbird_emotion_df_test) == len(
    test_emotion_scaled
)

assert np.array_equal(
    train_labels,
    bigbird_emotion_df_train[
        "label"
    ].to_numpy(dtype=np.int64)
)

assert np.array_equal(
    val_labels,
    bigbird_emotion_df_val[
        "label"
    ].to_numpy(dtype=np.int64)
)

assert np.array_equal(
    test_labels,
    bigbird_emotion_df_test[
        "label"
    ].to_numpy(dtype=np.int64)
)

print("✓ TRAIN users/features have equal row counts")
print("✓ VALIDATION users/features have equal row counts")
print("✓ TEST users/features have equal row counts")
print("✓ TRAIN labels aligned")
print("✓ VALIDATION labels aligned")
print("✓ TEST labels aligned")

print()
print("=" * 60)
print("FINAL VALUE INTEGRITY")
print("=" * 60)

for array, name in [
    (
        train_bigbird_emotion_metadata_scaled,
        "TRAIN metadata"
    ),
    (
        val_bigbird_emotion_metadata_scaled,
        "VAL metadata"
    ),
    (
        test_bigbird_emotion_metadata_scaled,
        "TEST metadata"
    ),
    (
        train_emotion_scaled,
        "TRAIN emotions"
    ),
    (
        val_emotion_scaled,
        "VAL emotions"
    ),
    (
        test_emotion_scaled,
        "TEST emotions"
    )
]:
    assert np.isfinite(array).all()
    print(
        f"✓ {name}: finite values only"
    )

print()
print("=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

train_distribution = np.bincount(
    train_labels,
    minlength=2
)

val_distribution = np.bincount(
    val_labels,
    minlength=2
)

test_distribution = np.bincount(
    test_labels,
    minlength=2
)

print("TRAIN:", train_distribution)
print("VALIDATION:", val_distribution)
print("TEST:", test_distribution)

assert np.array_equal(
    train_distribution,
    np.array([4122, 2013])
)

assert np.array_equal(
    val_distribution,
    np.array([515, 252])
)

assert np.array_equal(
    test_distribution,
    np.array([516, 251])
)

print()
print("=" * 60)
print("CELL 7 COMPLETE")
print("=" * 60)

print("✓ BigBird token inputs aligned")
print("✓ 53 metadata features aligned")
print("✓ 11 emotion features aligned")
print("✓ Labels aligned")
print("✓ Row counts verified")
print("✓ No NaN / Inf")
print("✓ Ready for training-only undersampling")

In [ ]:
# ============================================================
# CELL 8 — BIGBIRD + EMOTION-BASED
# ============================================================

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — TRAINING-ONLY RANDOM UNDERSAMPLING")
print("=" * 60)

print()
print("=" * 60)
print("ORIGINAL LABEL DISTRIBUTION")
print("=" * 60)

print()
print("TRAIN:")
print(
    bigbird_emotion_df_train["label"]
    .value_counts()
    .sort_index()
)

print()
print("VALIDATION:")
print(
    bigbird_emotion_df_val["label"]
    .value_counts()
    .sort_index()
)

print()
print("TEST:")
print(
    bigbird_emotion_df_test["label"]
    .value_counts()
    .sort_index()
)

positive_train = bigbird_emotion_df_train[
    bigbird_emotion_df_train["label"] == 1
].copy()

negative_train = bigbird_emotion_df_train[
    bigbird_emotion_df_train["label"] == 0
].copy()

print()
print("=" * 60)
print("TRAINING-ONLY UNDERSAMPLING")
print("=" * 60)

print(
    "Original positive users:",
    len(positive_train)
)

print(
    "Original negative users:",
    len(negative_train)
)

assert len(positive_train) == 2013
assert len(negative_train) == 4122

negative_sample = negative_train.sample(
    n=len(positive_train),
    random_state=SEED
)

balanced_train_bigbird_emotion_df = pd.concat(
    [
        positive_train,
        negative_sample
    ]
).sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print()
print("=" * 60)
print("BALANCED TRAINING DISTRIBUTION")
print("=" * 60)

balanced_counts = (
    balanced_train_bigbird_emotion_df["label"]
    .value_counts()
    .sort_index()
)

print(balanced_counts)

assert balanced_counts.to_dict() == {
    0: 2013,
    1: 2013
}

print()
print("=" * 60)
print("USER-LEVEL INTEGRITY")
print("=" * 60)

balanced_users = set(
    balanced_train_bigbird_emotion_df["user_id"]
)

val_users = set(
    bigbird_emotion_df_val["user_id"]
)

test_users = set(
    bigbird_emotion_df_test["user_id"]
)

print(
    "Balanced training users:",
    len(balanced_users)
)

print(
    "Unique balanced users:",
    balanced_train_bigbird_emotion_df[
        "user_id"
    ].nunique()
)

assert len(balanced_users) == 4026

assert (
    balanced_train_bigbird_emotion_df[
        "user_id"
    ].is_unique
)

assert len(balanced_users & val_users) == 0
assert len(balanced_users & test_users) == 0

print("✓ No duplicate training users")
print("✓ No train/validation overlap")
print("✓ No train/test overlap")

print()
print("=" * 60)
print("VALIDATION / TEST PRESERVATION")
print("=" * 60)

assert len(bigbird_emotion_df_val) == 767
assert len(bigbird_emotion_df_test) == 767

print(
    "Validation users:",
    len(bigbird_emotion_df_val)
)

print(
    "Test users:",
    len(bigbird_emotion_df_test)
)

print("✓ Validation unchanged")
print("✓ Test unchanged")

balanced_train_bigbird_emotion_df.to_pickle(
    os.path.join(
        RESULT_DIR,
        "balanced_train.pkl"
    )
)

print()
print("=" * 60)
print("CELL 8 COMPLETE")
print("=" * 60)

print("✓ Random undersampling applied to TRAIN ONLY")
print("✓ All 2,013 positive users retained")
print("✓ 2,013 negative users randomly selected")
print("✓ Balanced TRAIN = 4,026 users")
print("✓ VALIDATION = 767 users")
print("✓ TEST = 767 users")
print("✓ Random seed = 42")
print("✓ No cross-split user overlap")
print("✓ Balanced training split saved")

In [ ]:
# ============================================================
# CELL 9 — BIGBIRD + EMOTION-BASED
# ============================================================

from torch.utils.data import Dataset, DataLoader

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — BUILDING PYTORCH DATASETS")
print("=" * 60)

# Map original TRAIN users to their row positions
user_to_index = {
    user_id: i
    for i, user_id in enumerate(
        bigbird_emotion_df_train["user_id"]
    )
}

balanced_indices = np.array(
    [
        user_to_index[user_id]
        for user_id in
        balanced_train_bigbird_emotion_df["user_id"]
    ],
    dtype=np.int64
)

# Balanced TRAIN inputs
train_input_ids_balanced = train_input_ids[
    balanced_indices
]

train_attention_masks_balanced = (
    train_attention_masks[
        balanced_indices
    ]
)

train_metadata_balanced = (
    train_bigbird_emotion_metadata_scaled[
        balanced_indices
    ]
)

train_emotion_balanced = (
    train_emotion_scaled[
        balanced_indices
    ]
)

train_labels_balanced = (
    balanced_train_bigbird_emotion_df["label"]
    .to_numpy(dtype=np.int64)
)

# Validation labels
val_labels = (
    bigbird_emotion_df_val["label"]
    .to_numpy(dtype=np.int64)
)

# Test labels
test_labels = (
    bigbird_emotion_df_test["label"]
    .to_numpy(dtype=np.int64)
)


class EmotionBigBirdDataset(Dataset):

    def __init__(
        self,
        input_ids,
        attention_masks,
        metadata,
        emotions,
        labels
    ):
        self.input_ids = torch.tensor(
            input_ids,
            dtype=torch.long
        )

        self.attention_masks = torch.tensor(
            attention_masks,
            dtype=torch.long
        )

        self.metadata = torch.tensor(
            metadata,
            dtype=torch.float32
        )

        self.emotions = torch.tensor(
            emotions,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_masks[index],
            "metadata": self.metadata[index],
            "emotions": self.emotions[index],
            "labels": self.labels[index]
        }


train_dataset = EmotionBigBirdDataset(
    train_input_ids_balanced,
    train_attention_masks_balanced,
    train_metadata_balanced,
    train_emotion_balanced,
    train_labels_balanced
)

val_dataset = EmotionBigBirdDataset(
    val_input_ids,
    val_attention_masks,
    val_bigbird_emotion_metadata_scaled,
    val_emotion_scaled,
    val_labels
)

test_dataset = EmotionBigBirdDataset(
    test_input_ids,
    test_attention_masks,
    test_bigbird_emotion_metadata_scaled,
    test_emotion_scaled,
    test_labels
)


BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print()
print("=" * 60)
print("DATASET SIZES")
print("=" * 60)

print("Balanced Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

assert len(train_dataset) == 4026
assert len(val_dataset) == 767
assert len(test_dataset) == 767


print()
print("=" * 60)
print("DATALOADER CONFIGURATION")
print("=" * 60)

print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

assert len(train_loader) == 252
assert len(val_loader) == 48
assert len(test_loader) == 48


# Batch sanity check
batch = next(iter(train_loader))

print()
print("=" * 60)
print("BATCH SANITY CHECK")
print("=" * 60)

print(
    "Input IDs:",
    tuple(batch["input_ids"].shape)
)

print(
    "Attention mask:",
    tuple(batch["attention_mask"].shape)
)

print(
    "Metadata:",
    tuple(batch["metadata"].shape)
)

print(
    "Emotions:",
    tuple(batch["emotions"].shape)
)

print(
    "Labels:",
    tuple(batch["labels"].shape)
)

assert (
    tuple(batch["input_ids"].shape)
    == (16, 512)
)

assert (
    tuple(batch["attention_mask"].shape)
    == (16, 512)
)

assert (
    tuple(batch["metadata"].shape)
    == (16, 53)
)

assert (
    tuple(batch["emotions"].shape)
    == (16, 11)
)

assert (
    tuple(batch["labels"].shape)
    == (16,)
)


print()
print("=" * 60)
print("BATCH DATA TYPES")
print("=" * 60)

print(
    "Input IDs:",
    batch["input_ids"].dtype
)

print(
    "Attention mask:",
    batch["attention_mask"].dtype
)

print(
    "Metadata:",
    batch["metadata"].dtype
)

print(
    "Emotions:",
    batch["emotions"].dtype
)

print(
    "Labels:",
    batch["labels"].dtype
)

assert batch["input_ids"].dtype == torch.int64
assert batch["attention_mask"].dtype == torch.int64
assert batch["metadata"].dtype == torch.float32
assert batch["emotions"].dtype == torch.float32
assert batch["labels"].dtype == torch.int64


print()
print("=" * 60)
print("TRAINING LABEL BALANCE")
print("=" * 60)

balanced_distribution = np.bincount(
    train_labels_balanced,
    minlength=2
)

print(
    "Balanced TRAIN:",
    balanced_distribution
)

assert np.array_equal(
    balanced_distribution,
    np.array([2013, 2013])
)


print()
print("=" * 60)
print("USER ALIGNMENT")
print("=" * 60)

assert (
    len(
        balanced_train_bigbird_emotion_df
    )
    ==
    len(train_dataset)
)

assert (
    len(train_emotion_balanced)
    ==
    len(train_labels_balanced)
)

print(
    "Balanced training users:",
    len(balanced_train_bigbird_emotion_df)
)

print("✓ User alignment verified")


dataset_info = {
    "train_users": 4026,
    "validation_users": 767,
    "test_users": 767,
    "batch_size": 16,
    "train_batches": 252,
    "validation_batches": 48,
    "test_batches": 48,
    "context_length": 512,
    "metadata_features": 53,
    "emotion_features": 11
}

with open(
    os.path.join(
        RESULT_DIR,
        "dataset_info.json"
    ),
    "w"
) as f:
    json.dump(
        dataset_info,
        f,
        indent=2
    )


print()
print("=" * 60)
print("CELL 9 COMPLETE")
print("=" * 60)

print("✓ Balanced TRAIN Dataset created")
print("✓ Validation Dataset created")
print("✓ Test Dataset created")
print("✓ Batch size = 16")
print("✓ Train batches = 252")
print("✓ Validation batches = 48")
print("✓ Test batches = 48")
print("✓ Input dimensions verified")
print("✓ Data types verified")
print("✓ Label balance verified")
print("✓ User alignment verified")
print("✓ Dataset information saved")

In [ ]:
# ============================================================
# CELL 10 — BIGBIRD + EMOTION-BASED
# ============================================================

from transformers import AutoModel
import torch.nn as nn

print("=" * 60)
print("BIGBIRD + EMOTION-BASED — BUILDING FROZEN BIGBIRD + METADATA + EMOTION MODEL")
print("=" * 60)

bigbird_encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    token=hf_token
).to(device)

for param in bigbird_encoder.parameters():
    param.requires_grad = False


class BigBirdEmotionClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        metadata_dim=53,
        emotion_dim=11
    ):
        super().__init__()

        self.encoder = encoder
        self.text_hidden_size = 768

        # Text attention pooling
        self.attention_projection = nn.Linear(
            768,
            256
        )

        self.attention_score = nn.Linear(
            256,
            1
        )

        # Metadata branch
        self.metadata_branch = nn.Sequential(
            nn.Linear(
                metadata_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                128,
                64
            ),
            nn.ReLU()
        )

        # Emotion branch
        self.emotion_branch = nn.Sequential(
            nn.Linear(
                emotion_dim,
                64
            ),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Learnable gates
        self.metadata_beta = nn.Parameter(
            torch.tensor(0.0)
        )

        self.emotion_beta = nn.Parameter(
            torch.tensor(0.0)
        )

        # 768 text + 64 metadata + 64 emotion
        self.classifier = nn.Sequential(
            nn.Linear(
                896,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                512,
                256
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                256,
                2
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        metadata,
        emotions
    ):

        # Frozen BigBird
        with torch.no_grad():

            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        hidden_states = outputs.last_hidden_state

        # Attention pooling
        projected = torch.tanh(
            self.attention_projection(
                hidden_states
            )
        )

        scores = self.attention_score(
            projected
        ).squeeze(-1)

        scores = scores.masked_fill(
            attention_mask == 0,
            -1e9
        )

        attention_weights = torch.softmax(
            scores,
            dim=1
        )

        text_representation = torch.sum(
            hidden_states *
            attention_weights.unsqueeze(-1),
            dim=1
        )

        # Metadata
        metadata_representation = (
            self.metadata_branch(metadata)
        )

        metadata_gate = torch.sigmoid(
            self.metadata_beta
        )

        gated_metadata = (
            metadata_gate *
            metadata_representation
        )

        # Emotion
        emotion_representation = (
            self.emotion_branch(emotions)
        )

        emotion_gate = torch.sigmoid(
            self.emotion_beta
        )

        gated_emotion = (
            emotion_gate *
            emotion_representation
        )

        # Fusion
        fused = torch.cat(
            [
                text_representation,
                gated_metadata,
                gated_emotion
            ],
            dim=1
        )

        logits = self.classifier(
            fused
        )

        return {
            "logits": logits,
            "text_representation":
                text_representation,
            "metadata_representation":
                metadata_representation,
            "emotion_representation":
                emotion_representation,
            "attention_weights":
                attention_weights,
            "metadata_gate":
                metadata_gate,
            "emotion_gate":
                emotion_gate
        }


model = BigBirdEmotionClassifier(
    bigbird_encoder,
    metadata_dim=53,
    emotion_dim=11
).to(device)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
)

print()
print("=" * 60)
print("PARAMETER COUNT")
print("=" * 60)

print(
    "Total parameters:",
    total_params
)

print(
    "Trainable parameters:",
    trainable_params
)

print(
    "Frozen parameters:",
    frozen_params
)

print(
    "Total parameters (M):",
    round(
        total_params / 1e6,
        3
    )
)

print(
    "Trainable parameters (M):",
    round(
        trainable_params / 1e6,
        3
    )
)

print(
    "Frozen parameters (M):",
    round(
        frozen_params / 1e6,
        3
    )
)

encoder_trainable = sum(
    p.numel()
    for p in model.encoder.parameters()
    if p.requires_grad
)

assert encoder_trainable == 0

print()
print(
    "✓ All BigBird backbone parameters frozen"
)


# ============================================================
# TRAINABLE COMPONENTS
# ============================================================

print()
print("=" * 60)
print("TRAINABLE COMPONENTS")
print("=" * 60)

for name, param in model.named_parameters():

    if param.requires_grad:
        print(
            "✓",
            name
        )


# ============================================================
# ARCHITECTURE
# ============================================================

print()
print("=" * 60)
print("ARCHITECTURE CONFIGURATION")
print("=" * 60)

print(
    "Transformer:",
    MODEL_NAME
)

print(
    "Base architecture: BigBird-RoBERTa-base"
)

print(
    "Text hidden size: 768"
)

print(
    "Attention projection: 256"
)

print(
    "Metadata: 53 -> 128 -> 64"
)

print(
    "Emotion: 11 -> 64"
)

print(
    "Metadata gate: g_m = sigmoid(beta_m)"
)

print(
    "Emotion gate: g_e = sigmoid(beta_e)"
)

print(
    "Initial metadata beta: 0.0"
)

print(
    "Initial emotion beta: 0.0"
)

print(
    "Initial metadata gate: 0.5"
)

print(
    "Initial emotion gate: 0.5"
)

print(
    "Fusion: 768 + 64 + 64 = 896"
)

print(
    "Classifier: 896 -> 512 -> 256 -> 2"
)

print(
    "Dropout: 0.3"
)

print(
    "Transformer frozen: True"
)


# ============================================================
# FORWARD PASS SANITY CHECK
# ============================================================

print()
print("=" * 60)
print("FORWARD PASS SANITY CHECK")
print("=" * 60)

model.eval()

batch = next(
    iter(train_loader)
)

input_ids = batch[
    "input_ids"
].to(device)

attention_mask = batch[
    "attention_mask"
].to(device)

metadata = batch[
    "metadata"
].to(device)

emotions = batch[
    "emotions"
].to(device)

with torch.no_grad():

    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        metadata=metadata,
        emotions=emotions
    )

print(
    "Input IDs:",
    tuple(
        input_ids.shape
    )
)

print(
    "Attention mask:",
    tuple(
        attention_mask.shape
    )
)

print(
    "Metadata:",
    tuple(
        metadata.shape
    )
)

print(
    "Emotions:",
    tuple(
        emotions.shape
    )
)

print(
    "Logits:",
    tuple(
        output["logits"].shape
    )
)

print(
    "Text representation:",
    tuple(
        output[
            "text_representation"
        ].shape
    )
)

print(
    "Metadata representation:",
    tuple(
        output[
            "metadata_representation"
        ].shape
    )
)

print(
    "Emotion representation:",
    tuple(
        output[
            "emotion_representation"
        ].shape
    )
)

print(
    "Attention weights:",
    tuple(
        output[
            "attention_weights"
        ].shape
    )
)

metadata_gate = float(
    output[
        "metadata_gate"
    ].item()
)

emotion_gate = float(
    output[
        "emotion_gate"
    ].item()
)

print(
    "Metadata gate:",
    metadata_gate
)

print(
    "Emotion gate:",
    emotion_gate
)

padding_attention = output[
    "attention_weights"
][attention_mask == 0]

max_padding_attention = (
    float(
        padding_attention.max().item()
    )
    if padding_attention.numel()
    else 0.0
)

print(
    "Maximum attention on padding:",
    max_padding_attention
)

assert output[
    "logits"
].shape == (16, 2)

assert output[
    "text_representation"
].shape == (16, 768)

assert output[
    "metadata_representation"
].shape == (16, 64)

assert output[
    "emotion_representation"
].shape == (16, 64)

assert output[
    "attention_weights"
].shape == (16, 512)

assert abs(
    metadata_gate - 0.5
) < 1e-6

assert abs(
    emotion_gate - 0.5
) < 1e-6

assert max_padding_attention == 0.0

assert encoder_trainable == 0

print()
print(
    "✓ Transformer freeze verified"
)

print(
    "✓ Metadata gate verified"
)

print(
    "✓ Emotion gate verified"
)


# ============================================================
# SAVE CONFIGURATION
# ============================================================

architecture_config = {
    "model_name": MODEL_NAME,
    "base_architecture": "BigBird-RoBERTa-base",
    "text_hidden_size": 768,
    "attention_projection": 256,
    "metadata_branch": "53 -> 128 -> 64",
    "emotion_branch": "11 -> 64",
    "metadata_gate": "sigmoid(metadata_beta)",
    "emotion_gate": "sigmoid(emotion_beta)",
    "initial_metadata_beta": 0.0,
    "initial_emotion_beta": 0.0,
    "initial_metadata_gate": 0.5,
    "initial_emotion_gate": 0.5,
    "fusion": "768 + 64 + 64 = 896",
    "classifier": "896 -> 512 -> 256 -> 2",
    "dropout": 0.3,
    "transformer_frozen": True,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "frozen_parameters": frozen_params
}

with open(
    os.path.join(
        RESULT_DIR,
        "architecture_config.json"
    ),
    "w"
) as f:

    json.dump(
        architecture_config,
        f,
        indent=2
    )


print()
print("=" * 60)
print("CELL 10 COMPLETE")
print("=" * 60)

print(
    "✓ BigBird loaded"
)

print(
    "✓ Transformer backbone frozen"
)

print(
    "✓ Attention pooling active"
)

print(
    "✓ Metadata branch = 53 -> 128 -> 64"
)

print(
    "✓ Emotion branch = 11 -> 64"
)

print(
    "✓ Metadata gate active"
)

print(
    "✓ Emotion gate active"
)

print(
    "✓ Text representation = 768"
)

print(
    "✓ Fusion = 768 + 64 + 64 = 896"
)

print(
    "✓ Classifier = 896 -> 512 -> 256 -> 2"
)

print(
    "✓ Padding attention masked"
)

print(
    "✓ Forward pass successful"
)

print(
    "✓ Frozen BigBird + metadata + emotion model ready"
)

In [ ]:
# ============================================================
# CELL 11 — BIGBIRD + EMOTION-BASED
# ============================================================

import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

print("=" * 60)
print("STARTING BIGBIRD + EMOTION-BASED TRAINING")
print("=" * 60)

print()
print("Training users:", len(train_dataset))
print("Validation users:", len(val_dataset))
print("Test users:", len(test_dataset))
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Checkpoint metric: Validation F1")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Batch size:", BATCH_SIZE)
print("Transformer: FROZEN")

# Verify BigBird remains frozen
encoder_trainable = sum(
    p.numel()
    for p in model.encoder.parameters()
    if p.requires_grad
)

assert encoder_trainable == 0

print()
print("✓ BigBird backbone confirmed frozen")

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = (
    len(train_loader)
    * MAX_EPOCHS
)

warmup_steps = int(
    total_steps * WARMUP_RATIO
)


def lr_lambda(step):

    if step < warmup_steps:
        return (
            (step + 1)
            / max(1, warmup_steps)
        )

    return max(
        0.0,
        (total_steps - step)
        / max(
            1,
            total_steps - warmup_steps
        )
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

criterion = nn.CrossEntropyLoss()


def evaluate_model(model, loader):

    model.eval()

    total_loss = 0.0
    labels_all = []
    preds_all = []
    probs_all = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            metadata = batch[
                "metadata"
            ].to(device)

            emotions = batch[
                "emotions"
            ].to(device)

            labels = batch[
                "labels"
            ].to(device)

            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                metadata=metadata,
                emotions=emotions
            )

            logits = output["logits"]

            loss = criterion(
                logits,
                labels
            )

            total_loss += (
                loss.item()
                * labels.size(0)
            )

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            preds = torch.argmax(
                logits,
                dim=1
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            probs_all.extend(
                probs.cpu().numpy()
            )

    return {
        "loss":
            total_loss
            / len(loader.dataset),

        "accuracy":
            accuracy_score(
                labels_all,
                preds_all
            ),

        "precision":
            precision_score(
                labels_all,
                preds_all,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels_all,
                preds_all,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels_all,
                preds_all,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                labels_all,
                probs_all
            )
    }


history = []

best_val_f1 = -np.inf
best_epoch = None
best_state_dict = None

epochs_without_improvement = 0

training_start = time.time()


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()

    print()
    print("=" * 60)
    print(
        f"EPOCH {epoch}/{MAX_EPOCHS}"
    )
    print("=" * 60)

    model.train()

    running_loss = 0.0
    train_labels = []
    train_preds = []

    for batch_idx, batch in enumerate(
        train_loader,
        start=1
    ):

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        metadata = batch[
            "metadata"
        ].to(device)

        emotions = batch[
            "emotions"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        optimizer.zero_grad()

        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            metadata=metadata,
            emotions=emotions
        )

        logits = output["logits"]

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )

        preds = torch.argmax(
            logits.detach(),
            dim=1
        )

        train_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        train_preds.extend(
            preds.cpu().numpy()
        )

        if (
            batch_idx == 1
            or batch_idx % 50 == 0
            or batch_idx == len(train_loader)
        ):

            batch_accuracy = accuracy_score(
                labels.detach()
                .cpu()
                .numpy(),
                preds.cpu().numpy()
            )

            elapsed = (
                time.time()
                - epoch_start
            )

            print(
                f"Epoch {epoch} | "
                f"Batch "
                f"{batch_idx}/"
                f"{len(train_loader)} | "
                f"Loss: "
                f"{loss.item():.4f} | "
                f"Accuracy: "
                f"{batch_accuracy:.4f} | "
                f"LR: "
                f"{optimizer.param_groups[0]['lr']:.2e} | "
                f"Time: "
                f"{elapsed:.1f}s"
            )

    train_loss = (
        running_loss
        / len(train_loader.dataset)
    )

    train_accuracy = accuracy_score(
        train_labels,
        train_preds
    )

    train_precision = precision_score(
        train_labels,
        train_preds,
        zero_division=0
    )

    train_recall = recall_score(
        train_labels,
        train_preds,
        zero_division=0
    )

    train_f1 = f1_score(
        train_labels,
        train_preds,
        zero_division=0
    )

    print()
    print(
        f"Epoch {epoch} — Validation"
    )

    val_metrics = evaluate_model(
        model,
        val_loader
    )

    with torch.no_grad():

        metadata_gate = float(
            torch.sigmoid(
                model.metadata_beta
            ).item()
        )

        emotion_gate = float(
            torch.sigmoid(
                model.emotion_beta
            ).item()
        )

    epoch_time = (
        time.time()
        - epoch_start
    )

    print()
    print("=" * 60)
    print(f"EPOCH {epoch} SUMMARY")
    print("=" * 60)

    print(
        f"Train Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy: "
        f"{train_accuracy:.4f}"
    )

    print(
        f"Train Precision: "
        f"{train_precision:.4f}"
    )

    print(
        f"Train Recall: "
        f"{train_recall:.4f}"
    )

    print(
        f"Train F1: "
        f"{train_f1:.4f}"
    )

    print()
    print(
        f"Val Loss: "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"Val Accuracy: "
        f"{val_metrics['accuracy']:.4f}"
    )

    print(
        f"Val Precision: "
        f"{val_metrics['precision']:.4f}"
    )

    print(
        f"Val Recall: "
        f"{val_metrics['recall']:.4f}"
    )

    print(
        f"Val F1: "
        f"{val_metrics['f1']:.4f}"
    )

    print(
        f"Val ROC-AUC: "
        f"{val_metrics['roc_auc']:.4f}"
    )

    print(
        f"Metadata gate: "
        f"{metadata_gate:.6f}"
    )

    print(
        f"Emotion gate: "
        f"{emotion_gate:.6f}"
    )

    print(
        f"Epoch Time: "
        f"{epoch_time:.2f}s"
    )

    history.append({
        "epoch": epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_accuracy,

        "train_precision":
            train_precision,

        "train_recall":
            train_recall,

        "train_f1":
            train_f1,

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_precision":
            val_metrics["precision"],

        "val_recall":
            val_metrics["recall"],

        "val_f1":
            val_metrics["f1"],

        "val_roc_auc":
            val_metrics["roc_auc"],

        "metadata_gate":
            metadata_gate,

        "emotion_gate":
            emotion_gate,

        "learning_rate":
            optimizer.param_groups[0]["lr"],

        "epoch_time":
            epoch_time
    })

    # --------------------------------------------------------
    # BEST CHECKPOINT
    # --------------------------------------------------------

    if val_metrics["f1"] > best_val_f1:

        best_val_f1 = (
            val_metrics["f1"]
        )

        best_epoch = epoch

        best_state_dict = {
            key:
                value.detach()
                .cpu()
                .clone()
            for key, value
            in model.state_dict().items()
        }

        epochs_without_improvement = 0

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            "best_bigbird_emotion.pt"
        )

        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    best_state_dict,

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_f1":
                    best_val_f1,

                "experiment_name":
                    EXPERIMENT_NAME,

                "model_name":
                    MODEL_NAME,

                "cleaning_method":
                    CLEANING_METHOD
            },
            checkpoint_path
        )

        print()
        print("✓ NEW BEST CHECKPOINT")
        print(
            "Best Validation F1:",
            f"{best_val_f1:.4f}"
        )

        print(
            "Saved:",
            checkpoint_path
        )

    else:

        epochs_without_improvement += 1

        print()
        print(
            "No validation F1 improvement."
        )

        print(
            "Epochs without improvement:",
            f"{epochs_without_improvement} / "
            f"{EARLY_STOPPING_PATIENCE}"
        )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print()
        print(
            "EARLY STOPPING TRIGGERED"
        )

        break


assert best_state_dict is not None
assert best_epoch is not None

model.load_state_dict(
    best_state_dict
)

model.to(device)
model.eval()

total_training_time = (
    time.time()
    - training_start
)

history_df = pd.DataFrame(
    history
)

history_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

with torch.no_grad():

    learned_metadata_beta = float(
        model.metadata_beta.item()
    )

    learned_emotion_beta = float(
        model.emotion_beta.item()
    )

    learned_metadata_gate = float(
        torch.sigmoid(
            model.metadata_beta
        ).item()
    )

    learned_emotion_gate = float(
        torch.sigmoid(
            model.emotion_beta
        ).item()
    )


print()
print("=" * 60)
print("BIGBIRD + EMOTION-BASED TRAINING COMPLETE")
print("=" * 60)

print(
    "Epochs completed:",
    len(history)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Validation F1:",
    f"{best_val_f1:.4f}"
)

print(
    "Total training time:",
    f"{total_training_time:.2f} seconds"
)

print(
    "Best checkpoint:",
    os.path.join(
        CHECKPOINT_DIR,
        "best_bigbird_emotion.pt"
    )
)

print(
    "Training history saved:",
    history_path
)

print()
print(
    "Learned metadata beta:",
    learned_metadata_beta
)

print(
    "Learned emotion beta:",
    learned_emotion_beta
)

print(
    "Metadata gate:",
    f"{learned_metadata_gate:.6f}"
)

print(
    "Emotion gate:",
    f"{learned_emotion_gate:.6f}"
)

print()
print("✓ BigBird transformer remained frozen")
print("✓ Metadata branch trained")
print("✓ Emotion branch trained")
print("✓ Metadata gate learned")
print("✓ Emotion gate learned")

print()
print("=" * 60)
print("CELL 11 COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 12 — BIGBIRD + EMOTION-BASED
# FINAL TEST EVALUATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


print("=" * 60)
print("BIGBIRD + EMOTION-BASED — FINAL TEST EVALUATION")
print("=" * 60)


# ============================================================
# BEST CHECKPOINT
# ============================================================

best_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    "best_bigbird_emotion.pt"
)

assert os.path.exists(
    best_checkpoint
), f"Checkpoint not found: {best_checkpoint}"

print()
print("✓ Best BigBird checkpoint found")


# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

print()
print(
    "Loading BEST BigBird Emotion-Based checkpoint..."
)

checkpoint = torch.load(
    best_checkpoint,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


best_epoch_loaded = int(
    checkpoint["epoch"]
)

best_val_f1_loaded = float(
    checkpoint["best_val_f1"]
)


print()
print("✓ Best checkpoint loaded")

print(
    "Best epoch:",
    best_epoch_loaded
)

print(
    "Best validation F1:",
    f"{best_val_f1_loaded:.4f}"
)

print(
    "Checkpoint:",
    best_checkpoint
)


# ============================================================
# GATE VERIFICATION
# ============================================================

print()
print("=" * 60)
print("GATE VERIFICATION")
print("=" * 60)


learned_metadata_beta = float(
    model.metadata_beta
    .detach()
    .cpu()
)

learned_emotion_beta = float(
    model.emotion_beta
    .detach()
    .cpu()
)


metadata_gate = float(
    torch.sigmoid(
        model.metadata_beta
    )
    .detach()
    .cpu()
)

emotion_gate = float(
    torch.sigmoid(
        model.emotion_beta
    )
    .detach()
    .cpu()
)


print(
    "Learned metadata beta:",
    learned_metadata_beta
)

print(
    "Metadata gate:",
    f"{metadata_gate:.6f}"
)

print(
    "Learned emotion beta:",
    learned_emotion_beta
)

print(
    "Emotion gate:",
    f"{emotion_gate:.6f}"
)


# ============================================================
# FROZEN BACKBONE VERIFICATION
# ============================================================

encoder_trainable = sum(
    parameter.numel()
    for parameter in model.encoder.parameters()
    if parameter.requires_grad
)

assert encoder_trainable == 0

print()
print(
    "✓ BigBird backbone remains frozen"
)


# ============================================================
# TEST SET INTEGRITY
# ============================================================

expected_test_users = 767

actual_test_users = len(
    test_dataset
)

print()
print("=" * 60)
print("TEST SET INTEGRITY")
print("=" * 60)

print(
    "Expected test users:",
    expected_test_users
)

print(
    "Actual test users:",
    actual_test_users
)

assert (
    actual_test_users
    ==
    expected_test_users
)

print(
    "✓ Test set size correct"
)


# ============================================================
# EVALUATING TEST SET
# ============================================================

print()
print("=" * 60)
print("EVALUATING TEST SET")
print("=" * 60)


criterion = torch.nn.CrossEntropyLoss()

test_losses = []

test_labels_all = []
test_probs_all = []
test_predictions_all = []

sample_count = 0


with torch.no_grad():

    for batch_idx, batch in enumerate(
        test_loader,
        start=1
    ):

        input_ids = (
            batch[
                "input_ids"
            ]
            .to(device)
        )

        attention_mask = (
            batch[
                "attention_mask"
            ]
            .to(device)
        )

        metadata = (
            batch[
                "metadata"
            ]
            .to(device)
        )

        emotions = (
            batch[
                "emotions"
            ]
            .to(device)
        )

        labels = (
            batch[
                "labels"
            ]
            .to(device)
        )


        # ----------------------------------------------------
        # MODEL FORWARD
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            metadata=metadata,
            emotions=emotions
        )

        logits = outputs[
            "logits"
        ]


        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = criterion(
            logits,
            labels
        )

        test_losses.append(
            float(
                loss
                .detach()
                .cpu()
            )
        )


        # ----------------------------------------------------
        # POSITIVE CLASS PROBABILITY
        # ----------------------------------------------------

        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )[:, 1]


        # ----------------------------------------------------
        # BINARY PREDICTION
        # ----------------------------------------------------

        predictions = (
            probabilities >= 0.5
        ).long()


        # ----------------------------------------------------
        # COLLECT
        # ----------------------------------------------------

        test_labels_all.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        test_probs_all.extend(
            probabilities
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        test_predictions_all.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        sample_count += len(
            labels
        )


        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if (
            batch_idx == 1
            or batch_idx % 10 == 0
            or batch_idx
            == len(test_loader)
        ):

            print(
                f"Test batch "
                f"{batch_idx}/"
                f"{len(test_loader)}"
            )


# ============================================================
# TEST METRICS
# ============================================================

test_loss = float(
    np.mean(
        test_losses
    )
)

test_accuracy = accuracy_score(
    test_labels_all,
    test_predictions_all
)

test_precision = precision_score(
    test_labels_all,
    test_predictions_all,
    zero_division=0
)

test_recall = recall_score(
    test_labels_all,
    test_predictions_all,
    zero_division=0
)

test_f1 = f1_score(
    test_labels_all,
    test_predictions_all,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    test_labels_all,
    test_probs_all
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    test_labels_all,
    test_predictions_all,
    labels=[
        0,
        1
    ]
)

tn, fp, fn, tp = cm.ravel()


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

classification_report_text = (
    classification_report(
        test_labels_all,
        test_predictions_all,
        target_names=[
            "Negative",
            "Positive"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# FINAL TEST RESULTS
# ============================================================

print()
print("=" * 60)
print(
    "BIGBIRD + EMOTION-BASED — FINAL TEST RESULTS"
)
print("=" * 60)

print(
    "Test Loss:",
    f"{test_loss:.4f}"
)

print(
    "Test Accuracy:",
    f"{test_accuracy * 100:.2f}%"
)

print(
    "Test Precision:",
    f"{test_precision * 100:.2f}%"
)

print(
    "Test Recall:",
    f"{test_recall * 100:.2f}%"
)

print(
    "Test F1:",
    f"{test_f1 * 100:.2f}%"
)

print(
    "Test ROC-AUC:",
    f"{test_roc_auc * 100:.2f}%"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print()
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(
    "                 Predicted"
)

print(
    "                 0       1"
)

print(
    f"Actual 0        "
    f"{tn:<7}"
    f"{fp}"
)

print(
    f"Actual 1        "
    f"{fn:<7}"
    f"{tp}"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print()
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report_text
)


# ============================================================
# FINAL TEST INTEGRITY
# ============================================================

print()
print("=" * 60)
print("FINAL TEST INTEGRITY CHECK")
print("=" * 60)

print(
    "Expected test users:",
    expected_test_users
)

print(
    "Evaluated test users:",
    sample_count
)

assert (
    sample_count
    ==
    expected_test_users
)

assert len(
    test_labels_all
) == expected_test_users

assert len(
    test_probs_all
) == expected_test_users

assert len(
    test_predictions_all
) == expected_test_users


print(
    "✓ All test users evaluated exactly once"
)

print(
    "✓ Predictions aligned with labels"
)

print(
    "✓ Best BigBird checkpoint used"
)

print(
    "✓ Training data not used for evaluation"
)

print(
    "✓ Validation data not used for evaluation"
)

print(
    "✓ BigBird backbone remained frozen"
)


# ============================================================
# SAVE TEST PREDICTIONS
# ============================================================

predictions_df = pd.DataFrame({

    "user_id":
        test_df[
            "user_id"
        ].values,

    "username":
        test_df[
            "username"
        ].values,

    "label":
        np.asarray(
            test_labels_all
        ),

    "probability":
        np.asarray(
            test_probs_all
        ),

    "prediction":
        np.asarray(
            test_predictions_all
        )
})


predictions_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_test_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)


# ============================================================
# MODEL PARAMETER COUNTS
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_parameters = (
    total_parameters
    -
    trainable_parameters
)


# ============================================================
# SAVE TEST METRICS
# ============================================================

metrics_df = pd.DataFrame([{

    "Model Name":
        "BigBird",

    "Cleaning Method":
        "Emotion-Based",

    "Test Loss":
        test_loss,

    "Accuracy (%)":
        test_accuracy * 100,

    "Precision (%)":
        test_precision * 100,

    "Recall (%)":
        test_recall * 100,

    "F1 (%)":
        test_f1 * 100,

    "ROC-AUC (%)":
        test_roc_auc * 100,

    "Best Epoch":
        best_epoch_loaded,

    "Best Val F1 (%)":
        best_val_f1_loaded * 100,

    "Parameters (M)":
        total_parameters / 1e6,

    "Trainable Parameters (M)":
        trainable_parameters / 1e6,

    "Frozen Parameters (M)":
        frozen_parameters / 1e6,

    "TN":
        int(tn),

    "FP":
        int(fp),

    "FN":
        int(fn),

    "TP":
        int(tp),

    "Metadata Beta":
        learned_metadata_beta,

    "Metadata Gate":
        metadata_gate,

    "Emotion Beta":
        learned_emotion_beta,

    "Emotion Gate":
        emotion_gate

}])


metrics_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_test_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)


# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual 0",
        "Actual 1"
    ],
    columns=[
        "Predicted 0",
        "Predicted 1"
    ]
)


cm_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_confusion_matrix.csv"
)

cm_df.to_csv(
    cm_path
)


# ============================================================
# SAVE CLASSIFICATION REPORT
# ============================================================

report_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_classification_report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        classification_report_text
    )


# ============================================================
# SAVE COMPLETE FINAL RESULTS
# ============================================================

final_results = {

    "model_name":
        "BigBird",

    "model_checkpoint":
        MODEL_NAME,

    "cleaning_method":
        "Emotion-Based",

    "transformer_frozen":
        True,

    "test_loss":
        test_loss,

    "accuracy":
        test_accuracy,

    "precision":
        test_precision,

    "recall":
        test_recall,

    "f1":
        test_f1,

    "roc_auc":
        test_roc_auc,

    "best_epoch":
        best_epoch_loaded,

    "best_validation_f1":
        best_val_f1_loaded,

    "parameters_m":
        total_parameters / 1e6,

    "trainable_parameters_m":
        trainable_parameters / 1e6,

    "frozen_parameters_m":
        frozen_parameters / 1e6,

    "tn":
        int(tn),

    "fp":
        int(fp),

    "fn":
        int(fn),

    "tp":
        int(tp),

    "metadata_beta":
        learned_metadata_beta,

    "metadata_gate":
        metadata_gate,

    "emotion_beta":
        learned_emotion_beta,

    "emotion_gate":
        emotion_gate,

    "evaluated_test_users":
        sample_count
}


final_results_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_final_results.json"
)

with open(
    final_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_results,
        f,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 60)
print("CELL 12 COMPLETE")
print("=" * 60)

print(
    "✓ Best checkpoint evaluated"
)

print(
    "✓ Held-out test evaluated exactly once"
)

print(
    "✓ Final metrics calculated"
)

print(
    "✓ Predictions saved"
)

print(
    "✓ Metrics saved"
)

print(
    "✓ Confusion matrix saved"
)

print(
    "✓ Classification report saved"
)

print(
    "✓ Complete final results saved"
)

print(
    "✓ Frozen transformer verified"
)

In [ ]:
# ============================================================
# BIGBIRD + EMOTION-BASED — FINAL RESULT VISUALIZATION
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print("BIGBIRD + EMOTION-BASED — FINAL RESULT VISUALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

RESULT_DIR = "/kaggle/working/bigbird_emotion/results"

TEST_METRICS_PATH = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_test_metrics.csv"
)

HISTORY_PATH = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_training_history.csv"
)

CM_PATH = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_confusion_matrix.csv"
)

GRAPH_DIR = os.path.join(
    RESULT_DIR,
    "graphs"
)

os.makedirs(GRAPH_DIR, exist_ok=True)

# ------------------------------------------------------------
# LOAD RESULTS
# ------------------------------------------------------------

test_metrics = pd.read_csv(TEST_METRICS_PATH)
training_history = pd.read_csv(HISTORY_PATH)
confusion_matrix_df = pd.read_csv(CM_PATH, index_col=0)

print("\n✓ Test metrics loaded")
print("✓ Training history loaded")
print("✓ Confusion matrix loaded")

print("\nTest Metrics:")
display(test_metrics)

print("\nTraining History:")
display(training_history.tail())

print("\nConfusion Matrix:")
display(confusion_matrix_df)


# ============================================================
# EXTRACT EXACT SAVED COLUMN NAMES
# ============================================================

accuracy = test_metrics["Accuracy (%)"].iloc[0] / 100
precision = test_metrics["Precision (%)"].iloc[0] / 100
recall = test_metrics["Recall (%)"].iloc[0] / 100
f1 = test_metrics["F1 (%)"].iloc[0] / 100
roc_auc = test_metrics["ROC-AUC (%)"].iloc[0] / 100

print("\n✓ Test metrics extracted successfully")


# ============================================================
# GRAPH 1 — FINAL TEST METRICS
# ============================================================

plt.figure(figsize=(10, 6))

metrics_to_plot = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "ROC-AUC": roc_auc
}

bars = plt.bar(
    metrics_to_plot.keys(),
    metrics_to_plot.values()
)

plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("BigBird + Emotion-Based — Final Test Metrics")

for bar, value in zip(bars, metrics_to_plot.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()

graph1_path = os.path.join(
    GRAPH_DIR,
    "01_final_test_metrics.png"
)

plt.savefig(graph1_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 1 generated, displayed and saved: {graph1_path}")


# ============================================================
# GRAPH 2 — CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(7, 6))

sns.heatmap(
    confusion_matrix_df,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BigBird + Emotion-Based — Confusion Matrix")

plt.tight_layout()

graph2_path = os.path.join(
    GRAPH_DIR,
    "02_confusion_matrix.png"
)

plt.savefig(graph2_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 2 generated, displayed and saved: {graph2_path}")


# ============================================================
# GRAPH 3 — TRAINING VS VALIDATION LOSS
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    training_history["epoch"],
    training_history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BigBird + Emotion-Based — Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph3_path = os.path.join(
    GRAPH_DIR,
    "03_train_validation_loss.png"
)

plt.savefig(graph3_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 3 generated, displayed and saved: {graph3_path}")


# ============================================================
# GRAPH 4 — TRAINING VS VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    training_history["epoch"],
    training_history["val_accuracy"],
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("BigBird + Emotion-Based — Training vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph4_path = os.path.join(
    GRAPH_DIR,
    "04_train_validation_accuracy.png"
)

plt.savefig(graph4_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 4 generated, displayed and saved: {graph4_path}")


# ============================================================
# GRAPH 5 — TRAINING VS VALIDATION F1
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["train_f1"],
    marker="o",
    label="Training F1"
)

plt.plot(
    training_history["epoch"],
    training_history["val_f1"],
    marker="o",
    label="Validation F1"
)

plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("BigBird + Emotion-Based — Training vs Validation F1")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph5_path = os.path.join(
    GRAPH_DIR,
    "05_train_validation_f1.png"
)

plt.savefig(graph5_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 5 generated, displayed and saved: {graph5_path}")


# ============================================================
# GRAPH 6 — VALIDATION ROC-AUC
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["val_roc_auc"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("BigBird + Emotion-Based — Validation ROC-AUC")
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph6_path = os.path.join(
    GRAPH_DIR,
    "06_validation_roc_auc.png"
)

plt.savefig(graph6_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 6 generated, displayed and saved: {graph6_path}")


# ============================================================
# GRAPH 7 — VALIDATION PRECISION / RECALL
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["val_precision"],
    marker="o",
    label="Validation Precision"
)

plt.plot(
    training_history["epoch"],
    training_history["val_recall"],
    marker="o",
    label="Validation Recall"
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("BigBird + Emotion-Based — Validation Precision vs Recall")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph7_path = os.path.join(
    GRAPH_DIR,
    "07_validation_precision_recall.png"
)

plt.savefig(graph7_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 7 generated, displayed and saved: {graph7_path}")


# ============================================================
# GRAPH 8 — METADATA GATE
# ============================================================

best_row = training_history.loc[
    training_history["val_f1"].idxmax()
]

metadata_gate = best_row["metadata_gate"]

plt.figure(figsize=(7, 6))

bar = plt.bar(
    ["Metadata Gate"],
    [metadata_gate]
)

plt.ylim(0, 1)
plt.ylabel("Gate Value")
plt.title("BigBird + Emotion-Based — Learned Metadata Gate")

plt.text(
    bar[0].get_x() + bar[0].get_width() / 2,
    metadata_gate + 0.02,
    f"{metadata_gate:.3f}",
    ha="center"
)

plt.tight_layout()

graph8_path = os.path.join(
    GRAPH_DIR,
    "08_metadata_gate.png"
)

plt.savefig(graph8_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 8 generated, displayed and saved: {graph8_path}")


# ============================================================
# GRAPH 9 — LEARNING RATE SCHEDULE
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    training_history["epoch"],
    training_history["learning_rate"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("BigBird + Emotion-Based — Learning Rate Schedule")
plt.grid(True, alpha=0.3)

plt.tight_layout()

graph9_path = os.path.join(
    GRAPH_DIR,
    "09_learning_rate_schedule.png"
)

plt.savefig(graph9_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 9 generated, displayed and saved: {graph9_path}")


# ============================================================
# GRAPH 10 — POSITIVE CLASS PERFORMANCE
# ============================================================

positive_metrics = {
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}

plt.figure(figsize=(8, 6))

bars = plt.bar(
    positive_metrics.keys(),
    positive_metrics.values()
)

plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("BigBird + Emotion-Based — Positive Class Performance")

for bar, value in zip(bars, positive_metrics.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()

graph10_path = os.path.join(
    GRAPH_DIR,
    "10_positive_class_performance.png"
)

plt.savefig(graph10_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"✓ Graph 10 generated, displayed and saved: {graph10_path}")


# ============================================================
# GRAPH INDEX
# ============================================================

graph_index = pd.DataFrame({
    "graph_number": list(range(1, 11)),
    "graph_name": [
        "Final Test Metrics",
        "Confusion Matrix",
        "Train vs Validation Loss",
        "Train vs Validation Accuracy",
        "Train vs Validation F1",
        "Validation ROC-AUC",
        "Validation Precision and Recall",
        "Metadata Gate",
        "Learning Rate Schedule",
        "Positive Class Performance"
    ],
    "file": [
        graph1_path,
        graph2_path,
        graph3_path,
        graph4_path,
        graph5_path,
        graph6_path,
        graph7_path,
        graph8_path,
        graph9_path,
        graph10_path
    ]
})

graph_index_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_graph_index.csv"
)

graph_index.to_csv(
    graph_index_path,
    index=False
)

print("\n✓ Graph index saved")


# ============================================================
# FINAL SUMMARY
# ============================================================

best_epoch = int(
    training_history.loc[
        training_history["val_f1"].idxmax(),
        "epoch"
    ]
)

best_val_f1 = training_history["val_f1"].max()

summary = pd.DataFrame({
    "Model": ["BigBird"],
    "Cleaning_Method": ["Emotion-Based"],
    "Test_Accuracy": [accuracy],
    "Test_Precision": [precision],
    "Test_Recall": [recall],
    "Test_F1": [f1],
    "Test_ROC_AUC": [roc_auc],
    "Best_Validation_F1": [best_val_f1],
    "Best_Epoch": [best_epoch]
})

summary_path = os.path.join(
    RESULT_DIR,
    "bigbird_emotion_final_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 70)
print("GRAPH VISUALIZATION COMPLETE")
print("=" * 70)

print("✓ 10 graphs generated")
print("✓ All graphs displayed in notebook")
print("✓ All graphs saved as PNG")
print(f"✓ Graph folder: {GRAPH_DIR}")
print(f"✓ Graph index: {graph_index_path}")
print(f"✓ Final summary: {summary_path}")

display(summary)